# Faza 0: Smoke-Test Qwen-Image 2.1 na Kaggle (`GPU T4 x2`, `Internet: ON`)

Ten notebook weryfikuje:
1. Widocznosc obu kart T4 (`sm_75`), wolny dysk w `/tmp` oraz 29 GB RAM.
2. Instalacje ComfyUI w `/tmp/ComfyUI` + izolowany `/tmp/venv` przez `uv`.
3. Generacje `Turbo INT8` (`1024x1024` krotki i dlugi prompt, `1344x768`, oraz `Image Edit`) bez `--force-fp16`.
4. Izolowany test drugiej karty (`CUDA_VISIBLE_DEVICES=1`) bez `ComfyUI-MultiGPU`.
5. Zapis raportu `smoke_report.json` i obrazkow PNG do `/kaggle/working/smoke_test_results/`.


In [ ]:
import base64, os, pathlib, sys

PKG_ROOT = pathlib.Path(os.environ.get('QWEN_PKG_ROOT', '/tmp/qwen_pkg'))
PKG_DIR = PKG_ROOT / 'qwen_kaggle'
PKG_DIR.mkdir(parents=True, exist_ok=True)

_MODULES_B64 = {
  "__init__.py": "IiIiUXdlbi1JbWFnZSAyLjEgb24gS2FnZ2xlIChUNCB4Mikg4oCUIGNvcmUgcGFja2FnZS4iIiIKCl9fYWxsX18gPSBbCiAgICAiY29uc3RhbnRzIiwKICAgICJ3b3JrZmxvd3MiLAogICAgImJvb3RzdHJhcCIsCiAgICAic21va2VfdGVzdCIsCiAgICAicHJlcF9kYXRhc2V0IiwKXQo=",
  "constants.py": "IiIiQXV0aG9yaXRhdGl2ZSBjb25zdGFudHMsIG1vZGVsIGNhdGFsb2csIGFuZCBLYWdnbGUgcnVudGltZSBkZWZhdWx0cyBmb3IgUXdlbi1JbWFnZSAyLjEuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgTW9kZWxTcGVjOgogICAgIiIiTWV0YWRhdGEgZGVzY3JpYmluZyBhIHNpbmdsZSBRd2VuLUltYWdlIDIuMSBhc3NldCBpbiBDb21meS1PcmcvUXdlbi1JbWFnZS0yLjEuIiIiCgogICAga2V5OiBzdHIKICAgIGhmX3BhdGg6IHN0cgogICAgc3ViZGlyOiBzdHIKICAgIGZpbGVuYW1lOiBzdHIKICAgIGFwcHJveF9naWI6IGZsb2F0CiAgICBkZXNjcmlwdGlvbjogc3RyCgoKSEZfUkVQT19JRCA9ICJDb21meS1PcmcvUXdlbi1JbWFnZS0yLjEiCgpNT0RFTFM6IGRpY3Rbc3RyLCBNb2RlbFNwZWNdID0gewogICAgInVuZXRfdHVyYm8iOiBNb2RlbFNwZWMoCiAgICAgICAga2V5PSJ1bmV0X3R1cmJvIiwKICAgICAgICBoZl9wYXRoPSJkaWZmdXNpb25fbW9kZWxzL3F3ZW5faW1hZ2VfMi4xX3R1cmJvX2ludDhfY29udnJvdC5zYWZldGVuc29ycyIsCiAgICAgICAgc3ViZGlyPSJkaWZmdXNpb25fbW9kZWxzIiwKICAgICAgICBmaWxlbmFtZT0icXdlbl9pbWFnZV8yLjFfdHVyYm9faW50OF9jb252cm90LnNhZmV0ZW5zb3JzIiwKICAgICAgICBhcHByb3hfZ2liPTYuNzYsCiAgICAgICAgZGVzY3JpcHRpb249IlF3ZW4tSW1hZ2UgMi4xIFR1cmJvIFVOZXQgKElOVDggQ29udlJvdCwgOCBzdGVwcywgQ0ZHIDEuMCkiLAogICAgKSwKICAgICJ1bmV0X2Jhc2UiOiBNb2RlbFNwZWMoCiAgICAgICAga2V5PSJ1bmV0X2Jhc2UiLAogICAgICAgIGhmX3BhdGg9ImRpZmZ1c2lvbl9tb2RlbHMvcXdlbl9pbWFnZV8yLjFfaW50OF9jb252cm90LnNhZmV0ZW5zb3JzIiwKICAgICAgICBzdWJkaXI9ImRpZmZ1c2lvbl9tb2RlbHMiLAogICAgICAgIGZpbGVuYW1lPSJxd2VuX2ltYWdlXzIuMV9pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLAogICAgICAgIGFwcHJveF9naWI9Ni43NiwKICAgICAgICBkZXNjcmlwdGlvbj0iUXdlbi1JbWFnZSAyLjEgQmFzZSBVTmV0IChJTlQ4IENvbnZSb3QsIH4yNS00MCBzdGVwcykiLAogICAgKSwKICAgICJjbGlwX3ZsIjogTW9kZWxTcGVjKAogICAgICAgIGtleT0iY2xpcF92bCIsCiAgICAgICAgaGZfcGF0aD0idGV4dF9lbmNvZGVycy9xd2VuM3ZsXzhiX2ludDhfY29udnJvdC5zYWZldGVuc29ycyIsCiAgICAgICAgc3ViZGlyPSJ0ZXh0X2VuY29kZXJzIiwKICAgICAgICBmaWxlbmFtZT0icXdlbjN2bF84Yl9pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLAogICAgICAgIGFwcHJveF9naWI9OC43MSwKICAgICAgICBkZXNjcmlwdGlvbj0iUXdlbjMtVkwgOEIgdGV4dC92aXNpb24gZW5jb2RlciAoSU5UOCBDb252Um90KSIsCiAgICApLAogICAgInBlX2kyaSI6IE1vZGVsU3BlYygKICAgICAgICBrZXk9InBlX2kyaSIsCiAgICAgICAgaGZfcGF0aD0idGV4dF9lbmNvZGVycy9xd2VuMy41XzliX3F3ZW5faW1hZ2VfMi4xX3BlX2kyaS5pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLAogICAgICAgIHN1YmRpcj0idGV4dF9lbmNvZGVycyIsCiAgICAgICAgZmlsZW5hbWU9InF3ZW4zLjVfOWJfcXdlbl9pbWFnZV8yLjFfcGVfaTJpLmludDhfY29udnJvdC5zYWZldGVuc29ycyIsCiAgICAgICAgYXBwcm94X2dpYj04LjgyLAogICAgICAgIGRlc2NyaXB0aW9uPSJRd2VuMy41IDlCIFByb21wdCBFbmhhbmNlciBmb3IgSW1hZ2UgRWRpdCAoSU5UOCBDb252Um90KSIsCiAgICApLAogICAgInBlX3QyaSI6IE1vZGVsU3BlYygKICAgICAgICBrZXk9InBlX3QyaSIsCiAgICAgICAgaGZfcGF0aD0idGV4dF9lbmNvZGVycy9xd2VuMy41XzliX3F3ZW5faW1hZ2VfMi4xX3BlX3QyaS5pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLAogICAgICAgIHN1YmRpcj0idGV4dF9lbmNvZGVycyIsCiAgICAgICAgZmlsZW5hbWU9InF3ZW4zLjVfOWJfcXdlbl9pbWFnZV8yLjFfcGVfdDJpLmludDhfY29udnJvdC5zYWZldGVuc29ycyIsCiAgICAgICAgYXBwcm94X2dpYj04LjgyLAogICAgICAgIGRlc2NyaXB0aW9uPSJRd2VuMy41IDlCIFByb21wdCBFbmhhbmNlciBmb3IgVGV4dC10by1JbWFnZSAoSU5UOCBDb252Um90KSIsCiAgICApLAogICAgInZhZSI6IE1vZGVsU3BlYygKICAgICAgICBrZXk9InZhZSIsCiAgICAgICAgaGZfcGF0aD0idmFlL3F3ZW5faW1hZ2VfMi4xX3ZhZV9iZjE2LnNhZmV0ZW5zb3JzIiwKICAgICAgICBzdWJkaXI9InZhZSIsCiAgICAgICAgZmlsZW5hbWU9InF3ZW5faW1hZ2VfMi4xX3ZhZV9iZjE2LnNhZmV0ZW5zb3JzIiwKICAgICAgICBhcHByb3hfZ2liPTAuNjMsCiAgICAgICAgZGVzY3JpcHRpb249IlF3ZW4tSW1hZ2UgMi4xIFZBRSAoQkYxNikiLAogICAgKSwKICAgICJ1bmV0X3R1cmJvX2JmMTYiOiBNb2RlbFNwZWMoCiAgICAgICAga2V5PSJ1bmV0X3R1cmJvX2JmMTYiLAogICAgICAgIGhmX3BhdGg9ImRpZmZ1c2lvbl9tb2RlbHMvcXdlbl9pbWFnZV8yLjFfdHVyYm9fYmYxNi5zYWZldGVuc29ycyIsCiAgICAgICAgc3ViZGlyPSJkaWZmdXNpb25fbW9kZWxzIiwKICAgICAgICBmaWxlbmFtZT0icXdlbl9pbWFnZV8yLjFfdHVyYm9fYmYxNi5zYWZldGVuc29ycyIsCiAgICAgICAgYXBwcm94X2dpYj0xMy4yNSwKICAgICAgICBkZXNjcmlwdGlvbj0iUXdlbi1JbWFnZSAyLjEgVHVyYm8gVU5ldCAoTmF0aXZlIEJGMTYgZm9yIFRQVSB2NWUtOCwgOCBzdGVwcykiLAogICAgKSwKICAgICJ1bmV0X2Jhc2VfYmYxNiI6IE1vZGVsU3BlYygKICAgICAgICBrZXk9InVuZXRfYmFzZV9iZjE2IiwKICAgICAgICBoZl9wYXRoPSJkaWZmdXNpb25fbW9kZWxzL3F3ZW5faW1hZ2VfMi4xX2JmMTYuc2FmZXRlbnNvcnMiLAogICAgICAgIHN1YmRpcj0iZGlmZnVzaW9uX21vZGVscyIsCiAgICAgICAgZmlsZW5hbWU9InF3ZW5faW1hZ2VfMi4xX2JmMTYuc2FmZXRlbnNvcnMiLAogICAgICAgIGFwcHJveF9naWI9MTMuMjUsCiAgICAgICAgZGVzY3JpcHRpb249IlF3ZW4tSW1hZ2UgMi4xIEJhc2UgVU5ldCAoTmF0aXZlIEJGMTYgZm9yIFRQVSB2NWUtOCwgfjI1LTQwIHN0ZXBzKSIsCiAgICApLAogICAgImNsaXBfdmxfYmYxNiI6IE1vZGVsU3BlYygKICAgICAgICBrZXk9ImNsaXBfdmxfYmYxNiIsCiAgICAgICAgaGZfcGF0aD0idGV4dF9lbmNvZGVycy9xd2VuM3ZsXzhiX2JmMTYuc2FmZXRlbnNvcnMiLAogICAgICAgIHN1YmRpcj0idGV4dF9lbmNvZGVycyIsCiAgICAgICAgZmlsZW5hbWU9InF3ZW4zdmxfOGJfYmYxNi5zYWZldGVuc29ycyIsCiAgICAgICAgYXBwcm94X2dpYj0xNi4zMywKICAgICAgICBkZXNjcmlwdGlvbj0iUXdlbjMtVkwgOEIgdGV4dC92aXNpb24gZW5jb2RlciAoTmF0aXZlIEJGMTYgZm9yIFRQVSB2NWUtOCkiLAogICAgKSwKfQoKRklMRU5BTUVfVE9fU1BFQzogZGljdFtzdHIsIE1vZGVsU3BlY10gPSB7c3BlYy5maWxlbmFtZTogc3BlYyBmb3Igc3BlYyBpbiBNT0RFTFMudmFsdWVzKCl9CgpQUkVTRVRTOiBkaWN0W3N0ciwgdHVwbGVbc3RyLCAuLi5dXSA9IHsKICAgICMgRmFzdCBQaGFzZSAwIHNtb2tlIHRlc3QgKH4xNi4xIEdpQikKICAgICJzbW9rZSI6ICgidW5ldF90dXJibyIsICJjbGlwX3ZsIiwgInZhZSIpLAogICAgIyBDb3JlIGdlbmVyYXRpb24gd2l0aG91dCA5QiBwcm9tcHQgZW5oYW5jZXIgKH4yMi45IEdpQikKICAgICJjb3JlIjogKCJ1bmV0X3R1cmJvIiwgInVuZXRfYmFzZSIsICJjbGlwX3ZsIiwgInZhZSIpLAogICAgIyBGdWxsIGVkaXRpbmcgKyBnZW5lcmF0aW9uIHdpdGggSTJJIDlCIGVuaGFuY2VyICh+MzEuNyBHaUIpCiAgICAiZnVsbF9pMmkiOiAoInVuZXRfdHVyYm8iLCAidW5ldF9iYXNlIiwgImNsaXBfdmwiLCAicGVfaTJpIiwgInZhZSIpLAogICAgIyBDb21wbGV0ZSBzZXQgaW5jbHVkaW5nIGJvdGggVDJJIGFuZCBJMkkgOUIgZW5oYW5jZXJzICh+NDAuNSBHaUIpCiAgICAiYWxsIjogKCJ1bmV0X3R1cmJvIiwgInVuZXRfYmFzZSIsICJjbGlwX3ZsIiwgInBlX2kyaSIsICJwZV90MmkiLCAidmFlIiksCiAgICAjIE5hdGl2ZSBCRjE2IGNvcmUgc2V0IGZvciBUUFUgdjVlLTggMTI4IEdpQiBIQk0gKH40My41IEdpQikKICAgICJ0cHVfYmYxNl9jb3JlIjogKCJ1bmV0X3R1cmJvX2JmMTYiLCAidW5ldF9iYXNlX2JmMTYiLCAiY2xpcF92bF9iZjE2IiwgInZhZSIpLAogICAgIyBOYXRpdmUgQkYxNiBmdWxsIHNldCArIDlCIEkySSBlbmhhbmNlciBmb3IgVFBVIHY1ZS04ICh+NTIuMyBHaUIpCiAgICAidHB1X2JmMTYiOiAoInVuZXRfdHVyYm9fYmYxNiIsICJ1bmV0X2Jhc2VfYmYxNiIsICJjbGlwX3ZsX2JmMTYiLCAicGVfaTJpIiwgInZhZSIpLAp9CgpUQUVTRF9ERUNPREVSX1VSTCA9ICgKICAgICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vbWFkZWJ5b2xsaW4vdGFlc2QvbWFpbi90YWVxaTJfMV9kZWNvZGVyLnB0aCIKKQpUQUVTRF9ERUNPREVSX1NVQkRJUiA9ICJ2YWVfYXBwcm94IgpUQUVTRF9ERUNPREVSX0ZJTEVOQU1FID0gInRhZXFpMl8xX2RlY29kZXIucHRoIgoKV09SS0ZMT1dfVEVNUExBVEVfVVJMUzogZGljdFtzdHIsIHN0cl0gPSB7CiAgICAiaW1hZ2VfcXdlbl9pbWFnZV8yXzFfdDJpLmpzb24iOiAoCiAgICAgICAgImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS9Db21meS1Pcmcvd29ya2Zsb3dfdGVtcGxhdGVzL21haW4vdGVtcGxhdGVzLyIKICAgICAgICAiaW1hZ2VfcXdlbl9pbWFnZV8yXzFfdDJpLmpzb24iCiAgICApLAogICAgImltYWdlX3F3ZW5faW1hZ2VfMl8xX3R1cmJvX3QyaS5qc29uIjogKAogICAgICAgICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vQ29tZnktT3JnL3dvcmtmbG93X3RlbXBsYXRlcy9tYWluL3RlbXBsYXRlcy8iCiAgICAgICAgImltYWdlX3F3ZW5faW1hZ2VfMl8xX3R1cmJvX3QyaS5qc29uIgogICAgKSwKICAgICJpbWFnZV9xd2VuX2ltYWdlXzJfMV9pbWFnZV9lZGl0Lmpzb24iOiAoCiAgICAgICAgImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS9Db21meS1Pcmcvd29ya2Zsb3dfdGVtcGxhdGVzL21haW4vdGVtcGxhdGVzLyIKICAgICAgICAiaW1hZ2VfcXdlbl9pbWFnZV8yXzFfaW1hZ2VfZWRpdC5qc29uIgogICAgKSwKICAgICJpbWFnZV9xd2VuX2ltYWdlXzJfMV90dXJib19pbWFnZV9lZGl0Lmpzb24iOiAoCiAgICAgICAgImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS9Db21meS1Pcmcvd29ya2Zsb3dfdGVtcGxhdGVzL21haW4vdGVtcGxhdGVzLyIKICAgICAgICAiaW1hZ2VfcXdlbl9pbWFnZV8yXzFfdHVyYm9faW1hZ2VfZWRpdC5qc29uIgogICAgKSwKfQoKQ09NRllVSV9SRVBPX1VSTCA9ICJodHRwczovL2dpdGh1Yi5jb20vQ29tZnktT3JnL0NvbWZ5VUkuZ2l0IgpDTE9VREZMQVJFRF9MSU5VWF9VUkwgPSAoCiAgICAiaHR0cHM6Ly9naXRodWIuY29tL2Nsb3VkZmxhcmUvY2xvdWRmbGFyZWQvcmVsZWFzZXMvbGF0ZXN0L2Rvd25sb2FkL2Nsb3VkZmxhcmVkLWxpbnV4LWFtZDY0IgopClJVTlRJTUVfQ0FDSEVfU0xVRyA9ICJxd2VuLXQ0LXJ1bnRpbWUtY2FjaGUiClJVTlRJTUVfV0hFRUxTX0FSQ0hJVkUgPSAicXdlbl90NF93aGVlbHMudGFyIgpSVU5USU1FX0NPTUZZX0FSQ0hJVkUgPSAiY29tZnl1aV9yZXBvLnRhciIKUlVOVElNRV9DTE9VREZMQVJFRF9GSUxFTkFNRSA9ICJjbG91ZGZsYXJlZCIKUlVOVElNRV9NRVRBX0ZJTEVOQU1FID0gInF3ZW5fdDRfcnVudGltZV9tZXRhLmpzb24iCkdQVV9FWFRSQV9QSVBfUEFDS0FHRVM6IHR1cGxlW3N0ciwgLi4uXSA9ICgKICAgICJodWdnaW5nZmFjZS1odWJbaGZfeGV0XT49MC4zMi4wIiwKICAgICJjb21meS1raXRjaGVuIiwKICAgICJhaW9odHRwIiwKICAgICJwaWxsb3ciLAogICAgIm51bXB5IiwKICAgICJwYWNrYWdpbmciLAogICAgInJlcXVlc3RzIiwKICAgICJ3cmFwdCIsCikKREVGQVVMVF9DT01GWV9ESVIgPSBQYXRoKCIvdG1wL0NvbWZ5VUkiKQpERUZBVUxUX1ZFTlZfRElSID0gUGF0aCgiL3RtcC92ZW52IikKREVGQVVMVF9IRl9DQUNIRV9ESVIgPSBQYXRoKCIvdG1wL2hmX2NhY2hlIikKREVGQVVMVF9XT1JLSU5HX0RJUiA9IFBhdGgoIi9rYWdnbGUvd29ya2luZyIpCkRFRkFVTFRfSU5QVVRfRElSID0gUGF0aCgiL2thZ2dsZS9pbnB1dCIpCgpTVEFOREFSRF9NT0RFTF9TVUJESVJTOiB0dXBsZVtzdHIsIC4uLl0gPSAoCiAgICAiY2hlY2twb2ludHMiLAogICAgImNsaXAiLAogICAgImNvbnRyb2xuZXQiLAogICAgImRpZmZ1c2lvbl9tb2RlbHMiLAogICAgImVtYmVkZGluZ3MiLAogICAgImxvcmFzIiwKICAgICJtb2RlbF9wYXRjaGVzIiwKICAgICJ0ZXh0X2VuY29kZXJzIiwKICAgICJ1bmV0IiwKICAgICJ1cHNjYWxlX21vZGVscyIsCiAgICAidmFlIiwKICAgICJ2YWVfYXBwcm94IiwKKQoKIyBWZXJpZmllZCBmcm9tIG9mZmljaWFsIENvbWZ5LU9yZy93b3JrZmxvd190ZW1wbGF0ZXMgUXdlbi1JbWFnZSAyLjEgVHVyYm8gdGVtcGxhdGVzClRVUkJPX01BTlVBTF9TSUdNQVMgPSAoCiAgICAiMS4wLCAwLjk3ODQ1MywgMC45NTQxOCwgMC45MjY2MjYsIDAuODk1MDgsIDAuODQ1MTQ4LCAwLjcwNDUzNCwgMC40MTQ1NjgsIDAiCikKCiMgU2FmZSBmbGFncyBmb3IgS2FnZ2xlIFQ0ICgxNiBHQiBWUkFNIHBlciBjYXJkLCB+MzEgR0Igc3lzdGVtIFJBTSBhY3Jvc3MgMiBDb21meVVJIHdvcmtlcnMsIFR1cmluZyBzbV83NSkuCiMgRGVsaWJlcmF0ZWx5IG9taXRzIC0tZm9yY2UtZnAxNiB0byBhdm9pZCBGUDE2IE5hTiAvIGJsYWNrLWltYWdlIG92ZXJmbG93cyBvbiBRd2VuL1ZBRS4KIyBFeHBsaWNpdCAtLWNhY2hlLXJhbSB0aHJlc2hvbGRzIHByZXZlbnQgMiBwYXJhbGxlbCBDb21meVVJIHdvcmtlcnMgZnJvbSByYWNpbmcgb24gZGVmYXVsdCB+My4xIEdCIGZyZWUtUkFNIGV2aWN0aW9uLgpTQUZFX0NPTUZZX0ZMQUdTOiB0dXBsZVtzdHIsIC4uLl0gPSAoCiAgICAiLS1kaXNhYmxlLXBpbm5lZC1tZW1vcnkiLAogICAgIi0tZGlzYWJsZS1hc3luYy1vZmZsb2FkIiwKICAgICItLWNhY2hlLXJhbSIsCiAgICAiNi4wIiwKICAgICI4LjAiLAogICAgIi0tcHJldmlldy1tZXRob2QiLAogICAgInRhZXNkIiwKICAgICItLWRpc2FibGUtYXV0by1sYXVuY2giLAopCgpUUFVfU0FGRV9DT01GWV9GTEFHUzogdHVwbGVbc3RyLCAuLi5dID0gKAogICAgIi0tYmYxNi11bmV0IiwKICAgICItLWJmMTYtdmFlIiwKICAgICItLWRpc2FibGUtcGlubmVkLW1lbW9yeSIsCiAgICAiLS1kaXNhYmxlLWFzeW5jLW9mZmxvYWQiLAogICAgIi0tcHJldmlldy1tZXRob2QiLAogICAgInRhZXNkIiwKICAgICItLWRpc2FibGUtYXV0by1sYXVuY2giLAopCgpUUFVfQ09OVEVYVF9CVUNLRVRTOiB0dXBsZVtpbnQsIC4uLl0gPSAoMjU2LCA1MTIsIDEwMjQsIDIwNDgpCg==",
  "workflows.py": "IiIiQ29tZnlVSSBBUEkgcHJvbXB0IGdyYXBoIGJ1aWxkZXJzIGZvciBRd2VuLUltYWdlIDIuMSAoVDJJLCBFZGl0LCBUdXJibywgQmFzZSwgYW5kIFN0YWdlLTEgUEUpLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZAppbXBvcnQgcmUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSBxd2VuX2thZ2dsZS5jb25zdGFudHMgaW1wb3J0IE1PREVMUywgVFVSQk9fTUFOVUFMX1NJR01BUwoKTUFYX1JFRkVSRU5DRV9JTUFHRVMgPSAxNgpJREVOVElUWV9CTEVORF9NT0RFUyA9ICgibm9uZSIsICJ2ZWxvY2l0eV9ibGVuZCIsICJzdGVwX3N3YXAiKQpfTVVMVElfSU1BR0VfVEFHX0xJU1RfUkUgPSByZS5jb21waWxlKAogICAgciI8aW1hZ2VcZCs+KD86XHMqKD86LHwmfFwrfGFuZHxvcilccyo8aW1hZ2VcZCs+KSsiLAogICAgZmxhZ3M9cmUuSUdOT1JFQ0FTRSwKKQpfU0VDT05EQVJZX0lNQUdFX1RBR19SRSA9IHJlLmNvbXBpbGUociI8aW1hZ2UoPzpbMi05XXwxWzAtNl0pPiIsIGZsYWdzPXJlLklHTk9SRUNBU0UpCl9FSVRIRVJfUkVGX1JFID0gcmUuY29tcGlsZSgKICAgIHIiXGJmcm9tXHMrKD86ZWl0aGVyfGJvdGgpXHMrKD86c2lkZVxzK29mXHMrPGltYWdlMT58cmVmZXJlbmNlcz98cmVmZXJlbmNlXHMraW1hZ2VzPylcYiIsCiAgICBmbGFncz1yZS5JR05PUkVDQVNFLAopCl9CT1RIX1BBUkVOVFNfUE9TU19SRSA9IHJlLmNvbXBpbGUociJcYmJvdGhccysoPzpwYXJlbnRzfG1vdGhlcnN8ZmF0aGVycyknIiwgZmxhZ3M9cmUuSUdOT1JFQ0FTRSkKCgpkZWYgcHJlcGFyZV9zaW5nbGVfYnJhbmNoX3Byb21wdChwcm9tcHQ6IHN0cikgLT4gc3RyOgogICAgIiIiTm9ybWFsaXplIGEgbXVsdGktaW1hZ2UgZnVzaW9uIHByb21wdCBmb3IgYW4gaXNvbGF0ZWQgc2luZ2xlLXJlZmVyZW5jZSBgPGltYWdlMT5gIGJyYW5jaC4iIiIKICAgIGNsZWFuZWQgPSBwcm9tcHQuc3RyaXAoKQogICAgY2xlYW5lZCA9IF9NVUxUSV9JTUFHRV9UQUdfTElTVF9SRS5zdWIoIjxpbWFnZTE+IiwgY2xlYW5lZCkKICAgIGNsZWFuZWQgPSBfU0VDT05EQVJZX0lNQUdFX1RBR19SRS5zdWIoIjxpbWFnZTE+IiwgY2xlYW5lZCkKICAgIGNsZWFuZWQgPSBfRUlUSEVSX1JFRl9SRS5zdWIoImZyb20gPGltYWdlMT4iLCBjbGVhbmVkKQogICAgY2xlYW5lZCA9IF9CT1RIX1BBUkVOVFNfUE9TU19SRS5zdWIoIjxpbWFnZTE+J3MiLCBjbGVhbmVkKQogICAgaWYgIjxpbWFnZTE+IiBub3QgaW4gY2xlYW5lZC5sb3dlcigpOgogICAgICAgIGNsZWFuZWQgPSBmIlVzaW5nIHRoZSBmYWNpYWwgZmVhdHVyZXMgb2YgPGltYWdlMT46IHtjbGVhbmVkfSIgaWYgY2xlYW5lZCBlbHNlICJQb3J0cmFpdCBvZiB0aGUgcGVyc29uIGluIDxpbWFnZTE+LiIKICAgIHJldHVybiBjbGVhbmVkCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgR2VuZXJhdGlvblJlcXVlc3Q6CiAgICAiIiJOb3JtYWxpemVkIGdlbmVyYXRpb24gcGFyYW1ldGVycyBzaGFyZWQgYnkgc21va2UgdGVzdHMsIGJhdGNoIHJ1bnMsIGFuZCBBUEkgZ2F0ZXdheS4iIiIKCiAgICBwcm9tcHQ6IHN0cgogICAgbW9kZTogc3RyID0gInQyaSIgICMgInQyaSIgfCAiZWRpdCIKICAgIHByb2ZpbGU6IHN0ciA9ICJmYXN0IiAgIyAiZmFzdCIgKFR1cmJvKSB8ICJxdWFsaXR5IiAoQmFzZSkKICAgIG5lZ2F0aXZlX3Byb21wdDogc3RyID0gIiIKICAgIHdpZHRoOiBpbnQgPSAxMDI0CiAgICBoZWlnaHQ6IGludCA9IDEwMjQKICAgIHNlZWQ6IGludCA9IDQyNDI0MgogICAgc3RlcHM6IGludCB8IE5vbmUgPSBOb25lCiAgICBjZmc6IGZsb2F0IHwgTm9uZSA9IE5vbmUKICAgIHNhbXBsZXJfbmFtZTogc3RyID0gImV1bGVyIgogICAgc2NoZWR1bGVyOiBzdHIgPSAic2ltcGxlIgogICAgcmVzb2x1dGlvbjogaW50ID0gMTAyNAogICAgaW1hZ2VfbmFtZXM6IHR1cGxlW3N0ciwgLi4uXSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT10dXBsZSkKICAgIHVzZV9pbWFnZV9sYXRlbnQ6IGJvb2wgPSBUcnVlCiAgICBpZGVudGl0eV9ibGVuZF9tb2RlOiBzdHIgPSAibm9uZSIgICMgIm5vbmUiIHwgInZlbG9jaXR5X2JsZW5kIiB8ICJzdGVwX3N3YXAiCiAgICBpZGVudGl0eV9ibGVuZF9yYXRpbzogZmxvYXQgPSAwLjUgICMgd2VpZ2h0IG9mIDxpbWFnZTI+ICgwLjEuLjAuOSkgZm9yIDItaW1hZ2UgZnVzaW9uCiAgICBjYWNoZV9kZXZpY2U6IHN0ciA9ICJjcHUiICAjICJhdXRvIiB8ICJncHUiIHwgImNwdSIgfCAib2ZmIgogICAgY2FjaGVfZHR5cGU6IHN0ciA9ICJpbnQ4IiAgIyAiZGVmYXVsdCIgfCAiaW50OCIgfCAiaW50NCIKICAgIGZpbGVuYW1lX3ByZWZpeDogc3RyID0gInF3ZW4yMSIKICAgIHVuZXRfa2V5X292ZXJyaWRlOiBzdHIgfCBOb25lID0gTm9uZQogICAgY2xpcF9rZXlfb3ZlcnJpZGU6IHN0ciB8IE5vbmUgPSBOb25lCgoKZGVmIF92YWxpZGF0ZV9yZXF1ZXN0KHJlcTogR2VuZXJhdGlvblJlcXVlc3QpIC0+IE5vbmU6CiAgICBpZiByZXEubW9kZSBub3QgaW4gKCJ0MmkiLCAiZWRpdCIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbnN1cHBvcnRlZCBtb2RlPXtyZXEubW9kZSFyfTsgZXhwZWN0ZWQgJ3QyaScgb3IgJ2VkaXQnLiIpCiAgICBpZiByZXEucHJvZmlsZSBub3QgaW4gKCJmYXN0IiwgInF1YWxpdHkiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zdXBwb3J0ZWQgcHJvZmlsZT17cmVxLnByb2ZpbGUhcn07IGV4cGVjdGVkICdmYXN0JyBvciAncXVhbGl0eScuIikKICAgIGlmIHJlcS5pZGVudGl0eV9ibGVuZF9tb2RlIG5vdCBpbiBJREVOVElUWV9CTEVORF9NT0RFUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIkludmFsaWQgaWRlbnRpdHlfYmxlbmRfbW9kZT17cmVxLmlkZW50aXR5X2JsZW5kX21vZGUhcn07IGV4cGVjdGVkIG9uZSBvZiB7SURFTlRJVFlfQkxFTkRfTU9ERVN9LiIKICAgICAgICApCiAgICBpZiBub3QgKDAuMSA8PSBmbG9hdChyZXEuaWRlbnRpdHlfYmxlbmRfcmF0aW8pIDw9IDAuOSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaWRlbnRpdHlfYmxlbmRfcmF0aW8gbXVzdCBiZSBiZXR3ZWVuIDAuMSBhbmQgMC45LiIpCiAgICBpZiByZXEuY2FjaGVfZGV2aWNlIG5vdCBpbiAoImF1dG8iLCAiZ3B1IiwgImNwdSIsICJvZmYiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBjYWNoZV9kZXZpY2U9e3JlcS5jYWNoZV9kZXZpY2Uhcn0uIikKICAgIGlmIHJlcS5jYWNoZV9kdHlwZSBub3QgaW4gKCJkZWZhdWx0IiwgImludDgiLCAiaW50NCIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbnZhbGlkIGNhY2hlX2R0eXBlPXtyZXEuY2FjaGVfZHR5cGUhcn0uIikKICAgIGlmIHJlcS5tb2RlID09ICJlZGl0IiBhbmQgbm90IHJlcS5pbWFnZV9uYW1lczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFZGl0IG1vZGUgcmVxdWlyZXMgYXQgbGVhc3Qgb25lIGltYWdlIG5hbWUgaW4gaW1hZ2VfbmFtZXMuIikKICAgIGlmIGxlbihyZXEuaW1hZ2VfbmFtZXMpID4gTUFYX1JFRkVSRU5DRV9JTUFHRVM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlRvbyBtYW55IHJlZmVyZW5jZSBpbWFnZXMgKHtsZW4ocmVxLmltYWdlX25hbWVzKX0gPiB7TUFYX1JFRkVSRU5DRV9JTUFHRVN9KS4iKQogICAgaWYgcmVxLndpZHRoIDw9IDAgb3IgcmVxLmhlaWdodCA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbnZhbGlkIGRpbWVuc2lvbnM6IHtyZXEud2lkdGh9eHtyZXEuaGVpZ2h0fS4iKQoKCmRlZiBidWlsZF9xd2VuMjFfYXBpX3Byb21wdChyZXE6IEdlbmVyYXRpb25SZXF1ZXN0KSAtPiBkaWN0W3N0ciwgZGljdFtzdHIsIEFueV1dOgogICAgIiIiQnVpbGQgYSBDb21meVVJIEFQSS1mb3JtYXQgcHJvbXB0IGdyYXBoIHZlcmlmaWVkIGFnYWluc3QgQ29tZnktT3JnL0NvbWZ5VUkgbm9kZXMuIiIiCiAgICBfdmFsaWRhdGVfcmVxdWVzdChyZXEpCgogICAgZGVmYXVsdF91bmV0X2tleSA9ICJ1bmV0X3R1cmJvIiBpZiByZXEucHJvZmlsZSA9PSAiZmFzdCIgZWxzZSAidW5ldF9iYXNlIgogICAgdW5ldF9rZXkgPSByZXEudW5ldF9rZXlfb3ZlcnJpZGUgb3IgZGVmYXVsdF91bmV0X2tleQogICAgY2xpcF9rZXkgPSByZXEuY2xpcF9rZXlfb3ZlcnJpZGUgb3IgImNsaXBfdmwiCiAgICB1bmV0X25hbWUgPSBNT0RFTFNbdW5ldF9rZXldLmZpbGVuYW1lCiAgICBjbGlwX25hbWUgPSBNT0RFTFNbY2xpcF9rZXldLmZpbGVuYW1lCiAgICB2YWVfbmFtZSA9IE1PREVMU1sidmFlIl0uZmlsZW5hbWUKCiAgICBncmFwaDogZGljdFtzdHIsIGRpY3Rbc3RyLCBBbnldXSA9IHsKICAgICAgICAiMSI6IHsKICAgICAgICAgICAgImNsYXNzX3R5cGUiOiAiVU5FVExvYWRlciIsCiAgICAgICAgICAgICJpbnB1dHMiOiB7CiAgICAgICAgICAgICAgICAidW5ldF9uYW1lIjogdW5ldF9uYW1lLAogICAgICAgICAgICAgICAgIndlaWdodF9kdHlwZSI6ICJkZWZhdWx0IiwKICAgICAgICAgICAgfSwKICAgICAgICB9LAogICAgICAgICIyIjogewogICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJRd2VuSW1hZ2UyMUNhY2hlIiwKICAgICAgICAgICAgImlucHV0cyI6IHsKICAgICAgICAgICAgICAgICJtb2RlbCI6IFsiMSIsIDBdLAogICAgICAgICAgICAgICAgImRldmljZSI6IHJlcS5jYWNoZV9kZXZpY2UsCiAgICAgICAgICAgICAgICAiZHR5cGUiOiByZXEuY2FjaGVfZHR5cGUsCiAgICAgICAgICAgIH0sCiAgICAgICAgfSwKICAgICAgICAiMyI6IHsKICAgICAgICAgICAgImNsYXNzX3R5cGUiOiAiQ0xJUExvYWRlciIsCiAgICAgICAgICAgICJpbnB1dHMiOiB7CiAgICAgICAgICAgICAgICAiY2xpcF9uYW1lIjogY2xpcF9uYW1lLAogICAgICAgICAgICAgICAgInR5cGUiOiAicXdlbl9pbWFnZSIsCiAgICAgICAgICAgIH0sCiAgICAgICAgfSwKICAgICAgICAiNCI6IHsKICAgICAgICAgICAgImNsYXNzX3R5cGUiOiAiVkFFTG9hZGVyIiwKICAgICAgICAgICAgImlucHV0cyI6IHsKICAgICAgICAgICAgICAgICJ2YWVfbmFtZSI6IHZhZV9uYW1lLAogICAgICAgICAgICB9LAogICAgICAgIH0sCiAgICB9CgogICAgdXNlX2RpZmZ1c2lvbl9ibGVuZCA9ICgKICAgICAgICByZXEubW9kZSA9PSAiZWRpdCIKICAgICAgICBhbmQgbGVuKHJlcS5pbWFnZV9uYW1lcykgPj0gMgogICAgICAgIGFuZCByZXEuaWRlbnRpdHlfYmxlbmRfbW9kZSBpbiAoInZlbG9jaXR5X2JsZW5kIiwgInN0ZXBfc3dhcCIpCiAgICApCgogICAgaWYgdXNlX2RpZmZ1c2lvbl9ibGVuZDoKICAgICAgICBuX3JlZnMgPSBsZW4ocmVxLmltYWdlX25hbWVzKQogICAgICAgIGJyYW5jaF9wcm9tcHQgPSBwcmVwYXJlX3NpbmdsZV9icmFuY2hfcHJvbXB0KHJlcS5wcm9tcHQpCiAgICAgICAgcmF0aW8gPSByb3VuZChmbG9hdChyZXEuaWRlbnRpdHlfYmxlbmRfcmF0aW8pLCAzKQogICAgICAgIGlmIG5fcmVmcyA9PSAyOgogICAgICAgICAgICB3ZWlnaHRzID0gW3JvdW5kKDEuMCAtIHJhdGlvLCAzKSwgcmF0aW9dCiAgICAgICAgICAgIHJhbmdlcyA9IFsoMC4wLCByb3VuZCgxLjAgLSByYXRpbywgMykpLCAocm91bmQoMS4wIC0gcmF0aW8sIDMpLCAxLjApXQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHdlaWdodHMgPSBbcm91bmQoMS4wIC8gbl9yZWZzLCA0KSBmb3IgXyBpbiByYW5nZShuX3JlZnMpXQogICAgICAgICAgICByYW5nZXMgPSBbCiAgICAgICAgICAgICAgICAocm91bmQoaSAvIG5fcmVmcywgNCksIDEuMCBpZiBpID09IG5fcmVmcyAtIDEgZWxzZSByb3VuZCgoaSArIDEpIC8gbl9yZWZzLCA0KSkKICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKG5fcmVmcykKICAgICAgICAgICAgXQoKICAgICAgICBuZXh0X2JsZW5kX2lkID0gNTAKICAgICAgICBwb3NfYnJhbmNoZXM6IGxpc3RbbGlzdFtBbnldXSA9IFtdCiAgICAgICAgbmVnX2JyYW5jaGVzOiBsaXN0W2xpc3RbQW55XV0gPSBbXQoKICAgICAgICBmb3IgaWR4LCBpbWdfbmFtZSBpbiBlbnVtZXJhdGUocmVxLmltYWdlX25hbWVzKToKICAgICAgICAgICAgbG9hZF9pZCA9IHN0cigyMCArIGlkeCkKICAgICAgICAgICAgZ3JhcGhbbG9hZF9pZF0gPSB7CiAgICAgICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJMb2FkSW1hZ2UiLAogICAgICAgICAgICAgICAgImlucHV0cyI6IHsiaW1hZ2UiOiBpbWdfbmFtZX0sCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZW5jX2lkID0gc3RyKG5leHRfYmxlbmRfaWQpCiAgICAgICAgICAgIG5leHRfYmxlbmRfaWQgKz0gMQogICAgICAgICAgICBncmFwaFtlbmNfaWRdID0gewogICAgICAgICAgICAgICAgImNsYXNzX3R5cGUiOiAiVGV4dEVuY29kZVF3ZW5JbWFnZTIxIiwKICAgICAgICAgICAgICAgICJpbnB1dHMiOiB7CiAgICAgICAgICAgICAgICAgICAgImNsaXAiOiBbIjMiLCAwXSwKICAgICAgICAgICAgICAgICAgICAicHJvbXB0IjogYnJhbmNoX3Byb21wdCwKICAgICAgICAgICAgICAgICAgICAibmVnYXRpdmVfcHJvbXB0IjogcmVxLm5lZ2F0aXZlX3Byb21wdCwKICAgICAgICAgICAgICAgICAgICAicmVzb2x1dGlvbiI6IHJlcS5yZXNvbHV0aW9uLAogICAgICAgICAgICAgICAgICAgICJ2YWUiOiBbIjQiLCAwXSwKICAgICAgICAgICAgICAgICAgICAiaW1hZ2VzLmltYWdlXzEiOiBbbG9hZF9pZCwgMF0sCiAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICB9CgogICAgICAgICAgICBwb3NfbW9kX2lkID0gc3RyKG5leHRfYmxlbmRfaWQpCiAgICAgICAgICAgIG5leHRfYmxlbmRfaWQgKz0gMQogICAgICAgICAgICBuZWdfbW9kX2lkID0gc3RyKG5leHRfYmxlbmRfaWQpCiAgICAgICAgICAgIG5leHRfYmxlbmRfaWQgKz0gMQogICAgICAgICAgICBpZiByZXEuaWRlbnRpdHlfYmxlbmRfbW9kZSA9PSAidmVsb2NpdHlfYmxlbmQiOgogICAgICAgICAgICAgICAgc3RyZW5ndGggPSB3ZWlnaHRzW2lkeF0KICAgICAgICAgICAgICAgIGdyYXBoW3Bvc19tb2RfaWRdID0gewogICAgICAgICAgICAgICAgICAgICJjbGFzc190eXBlIjogIkNvbmRpdGlvbmluZ1NldEFyZWFTdHJlbmd0aCIsCiAgICAgICAgICAgICAgICAgICAgImlucHV0cyI6IHsiY29uZGl0aW9uaW5nIjogW2VuY19pZCwgMF0sICJzdHJlbmd0aCI6IHN0cmVuZ3RofSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGdyYXBoW25lZ19tb2RfaWRdID0gewogICAgICAgICAgICAgICAgICAgICJjbGFzc190eXBlIjogIkNvbmRpdGlvbmluZ1NldEFyZWFTdHJlbmd0aCIsCiAgICAgICAgICAgICAgICAgICAgImlucHV0cyI6IHsiY29uZGl0aW9uaW5nIjogW2VuY19pZCwgMV0sICJzdHJlbmd0aCI6IHN0cmVuZ3RofSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHN0YXJ0X3BjdCwgZW5kX3BjdCA9IHJhbmdlc1tpZHhdCiAgICAgICAgICAgICAgICBncmFwaFtwb3NfbW9kX2lkXSA9IHsKICAgICAgICAgICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJDb25kaXRpb25pbmdTZXRUaW1lc3RlcFJhbmdlIiwKICAgICAgICAgICAgICAgICAgICAiaW5wdXRzIjogeyJjb25kaXRpb25pbmciOiBbZW5jX2lkLCAwXSwgInN0YXJ0Ijogc3RhcnRfcGN0LCAiZW5kIjogZW5kX3BjdH0sCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICBncmFwaFtuZWdfbW9kX2lkXSA9IHsKICAgICAgICAgICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJDb25kaXRpb25pbmdTZXRUaW1lc3RlcFJhbmdlIiwKICAgICAgICAgICAgICAgICAgICAiaW5wdXRzIjogeyJjb25kaXRpb25pbmciOiBbZW5jX2lkLCAxXSwgInN0YXJ0Ijogc3RhcnRfcGN0LCAiZW5kIjogZW5kX3BjdH0sCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgIHBvc19icmFuY2hlcy5hcHBlbmQoW3Bvc19tb2RfaWQsIDBdKQogICAgICAgICAgICBuZWdfYnJhbmNoZXMuYXBwZW5kKFtuZWdfbW9kX2lkLCAwXSkKCiAgICAgICAgcG9zaXRpdmVfcmVmID0gcG9zX2JyYW5jaGVzWzBdCiAgICAgICAgZm9yIG54dCBpbiBwb3NfYnJhbmNoZXNbMTpdOgogICAgICAgICAgICBjb21iX2lkID0gc3RyKG5leHRfYmxlbmRfaWQpCiAgICAgICAgICAgIG5leHRfYmxlbmRfaWQgKz0gMQogICAgICAgICAgICBncmFwaFtjb21iX2lkXSA9IHsKICAgICAgICAgICAgICAgICJjbGFzc190eXBlIjogIkNvbmRpdGlvbmluZ0NvbWJpbmUiLAogICAgICAgICAgICAgICAgImlucHV0cyI6IHsiY29uZGl0aW9uaW5nXzEiOiBwb3NpdGl2ZV9yZWYsICJjb25kaXRpb25pbmdfMiI6IG54dH0sCiAgICAgICAgICAgIH0KICAgICAgICAgICAgcG9zaXRpdmVfcmVmID0gW2NvbWJfaWQsIDBdCgogICAgICAgIG5lZ2F0aXZlX3JlZiA9IG5lZ19icmFuY2hlc1swXQogICAgICAgIGZvciBueHQgaW4gbmVnX2JyYW5jaGVzWzE6XToKICAgICAgICAgICAgY29tYl9pZCA9IHN0cihuZXh0X2JsZW5kX2lkKQogICAgICAgICAgICBuZXh0X2JsZW5kX2lkICs9IDEKICAgICAgICAgICAgZ3JhcGhbY29tYl9pZF0gPSB7CiAgICAgICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJDb25kaXRpb25pbmdDb21iaW5lIiwKICAgICAgICAgICAgICAgICJpbnB1dHMiOiB7ImNvbmRpdGlvbmluZ18xIjogbmVnYXRpdmVfcmVmLCAiY29uZGl0aW9uaW5nXzIiOiBueHR9LAogICAgICAgICAgICB9CiAgICAgICAgICAgIG5lZ2F0aXZlX3JlZiA9IFtjb21iX2lkLCAwXQoKICAgICAgICAjIERpZmZ1c2lvbi1sZXZlbCBpZGVudGl0eSBmdXNpb24gYWx3YXlzIGdlbmVyYXRlcyBvbiBhIGZyZXNoIEVtcHR5TGF0ZW50SW1hZ2Ugc28gbmVpdGhlciByZWZlcmVuY2UgbG9ja3MgZnJhbWluZy4KICAgICAgICBncmFwaFsiNiJdID0gewogICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJFbXB0eUxhdGVudEltYWdlIiwKICAgICAgICAgICAgImlucHV0cyI6IHsKICAgICAgICAgICAgICAgICJ3aWR0aCI6IHJlcS53aWR0aCwKICAgICAgICAgICAgICAgICJoZWlnaHQiOiByZXEuaGVpZ2h0LAogICAgICAgICAgICAgICAgImJhdGNoX3NpemUiOiAxLAogICAgICAgICAgICB9LAogICAgICAgIH0KICAgICAgICBsYXRlbnRfcmVmOiBsaXN0W0FueV0gPSBbIjYiLCAwXQogICAgZWxzZToKICAgICAgICBlbmNvZGVfaW5wdXRzOiBkaWN0W3N0ciwgQW55XSA9IHsKICAgICAgICAgICAgImNsaXAiOiBbIjMiLCAwXSwKICAgICAgICAgICAgInByb21wdCI6IHJlcS5wcm9tcHQsCiAgICAgICAgICAgICJuZWdhdGl2ZV9wcm9tcHQiOiByZXEubmVnYXRpdmVfcHJvbXB0LAogICAgICAgICAgICAicmVzb2x1dGlvbiI6IHJlcS5yZXNvbHV0aW9uLAogICAgICAgIH0KCiAgICAgICAgaWYgcmVxLm1vZGUgPT0gImVkaXQiOgogICAgICAgICAgICBlbmNvZGVfaW5wdXRzWyJ2YWUiXSA9IFsiNCIsIDBdCiAgICAgICAgICAgIG5leHRfaWQgPSAyMAogICAgICAgICAgICBmb3IgaWR4LCBpbWdfbmFtZSBpbiBlbnVtZXJhdGUocmVxLmltYWdlX25hbWVzLCBzdGFydD0xKToKICAgICAgICAgICAgICAgIG5vZGVfaWQgPSBzdHIobmV4dF9pZCkKICAgICAgICAgICAgICAgIG5leHRfaWQgKz0gMQogICAgICAgICAgICAgICAgZ3JhcGhbbm9kZV9pZF0gPSB7CiAgICAgICAgICAgICAgICAgICAgImNsYXNzX3R5cGUiOiAiTG9hZEltYWdlIiwKICAgICAgICAgICAgICAgICAgICAiaW5wdXRzIjogeyJpbWFnZSI6IGltZ19uYW1lfSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGVuY29kZV9pbnB1dHNbZiJpbWFnZXMuaW1hZ2Vfe2lkeH0iXSA9IFtub2RlX2lkLCAwXQoKICAgICAgICBncmFwaFsiNSJdID0gewogICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJUZXh0RW5jb2RlUXdlbkltYWdlMjEiLAogICAgICAgICAgICAiaW5wdXRzIjogZW5jb2RlX2lucHV0cywKICAgICAgICB9CiAgICAgICAgcG9zaXRpdmVfcmVmID0gWyI1IiwgMF0KICAgICAgICBuZWdhdGl2ZV9yZWYgPSBbIjUiLCAxXQoKICAgICAgICBpZiByZXEubW9kZSA9PSAiZWRpdCIgYW5kIHJlcS51c2VfaW1hZ2VfbGF0ZW50OgogICAgICAgICAgICBsYXRlbnRfcmVmID0gWyI1IiwgMl0KICAgICAgICBlbHNlOgogICAgICAgICAgICBncmFwaFsiNiJdID0gewogICAgICAgICAgICAgICAgImNsYXNzX3R5cGUiOiAiRW1wdHlMYXRlbnRJbWFnZSIsCiAgICAgICAgICAgICAgICAiaW5wdXRzIjogewogICAgICAgICAgICAgICAgICAgICJ3aWR0aCI6IHJlcS53aWR0aCwKICAgICAgICAgICAgICAgICAgICAiaGVpZ2h0IjogcmVxLmhlaWdodCwKICAgICAgICAgICAgICAgICAgICAiYmF0Y2hfc2l6ZSI6IDEsCiAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICB9CiAgICAgICAgICAgIGxhdGVudF9yZWYgPSBbIjYiLCAwXQoKICAgIGlmIHJlcS5wcm9maWxlID09ICJmYXN0IjoKICAgICAgICBjZmdfdmFsID0gMS4wIGlmIHJlcS5jZmcgaXMgTm9uZSBlbHNlIGZsb2F0KHJlcS5jZmcpCiAgICAgICAgZ3JhcGhbIjciXSA9IHsKICAgICAgICAgICAgImNsYXNzX3R5cGUiOiAiUmFuZG9tTm9pc2UiLAogICAgICAgICAgICAiaW5wdXRzIjogeyJub2lzZV9zZWVkIjogaW50KHJlcS5zZWVkKX0sCiAgICAgICAgfQogICAgICAgIGdyYXBoWyI4Il0gPSB7CiAgICAgICAgICAgICJjbGFzc190eXBlIjogIktTYW1wbGVyU2VsZWN0IiwKICAgICAgICAgICAgImlucHV0cyI6IHsic2FtcGxlcl9uYW1lIjogcmVxLnNhbXBsZXJfbmFtZX0sCiAgICAgICAgfQogICAgICAgIGdyYXBoWyI5Il0gPSB7CiAgICAgICAgICAgICJjbGFzc190eXBlIjogIk1hbnVhbFNpZ21hcyIsCiAgICAgICAgICAgICJpbnB1dHMiOiB7InNpZ21hcyI6IFRVUkJPX01BTlVBTF9TSUdNQVN9LAogICAgICAgIH0KICAgICAgICBncmFwaFsiMTAiXSA9IHsKICAgICAgICAgICAgImNsYXNzX3R5cGUiOiAiQ0ZHR3VpZGVyIiwKICAgICAgICAgICAgImlucHV0cyI6IHsKICAgICAgICAgICAgICAgICJtb2RlbCI6IFsiMiIsIDBdLAogICAgICAgICAgICAgICAgInBvc2l0aXZlIjogcG9zaXRpdmVfcmVmLAogICAgICAgICAgICAgICAgIm5lZ2F0aXZlIjogbmVnYXRpdmVfcmVmLAogICAgICAgICAgICAgICAgImNmZyI6IGNmZ192YWwsCiAgICAgICAgICAgIH0sCiAgICAgICAgfQogICAgICAgIGdyYXBoWyIxMSJdID0gewogICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJTYW1wbGVyQ3VzdG9tQWR2YW5jZWQiLAogICAgICAgICAgICAiaW5wdXRzIjogewogICAgICAgICAgICAgICAgIm5vaXNlIjogWyI3IiwgMF0sCiAgICAgICAgICAgICAgICAiZ3VpZGVyIjogWyIxMCIsIDBdLAogICAgICAgICAgICAgICAgInNhbXBsZXIiOiBbIjgiLCAwXSwKICAgICAgICAgICAgICAgICJzaWdtYXMiOiBbIjkiLCAwXSwKICAgICAgICAgICAgICAgICJsYXRlbnRfaW1hZ2UiOiBsYXRlbnRfcmVmLAogICAgICAgICAgICB9LAogICAgICAgIH0KICAgICAgICBzYW1wbGVzX3JlZjogbGlzdFtBbnldID0gWyIxMSIsIDBdCiAgICBlbHNlOgogICAgICAgIHN0ZXBzX3ZhbCA9IDI1IGlmIHJlcS5zdGVwcyBpcyBOb25lIGVsc2UgaW50KHJlcS5zdGVwcykKICAgICAgICBjZmdfdmFsID0gMS4wIGlmIHJlcS5jZmcgaXMgTm9uZSBlbHNlIGZsb2F0KHJlcS5jZmcpCiAgICAgICAgZ3JhcGhbIjExIl0gPSB7CiAgICAgICAgICAgICJjbGFzc190eXBlIjogIktTYW1wbGVyIiwKICAgICAgICAgICAgImlucHV0cyI6IHsKICAgICAgICAgICAgICAgICJtb2RlbCI6IFsiMiIsIDBdLAogICAgICAgICAgICAgICAgInNlZWQiOiBpbnQocmVxLnNlZWQpLAogICAgICAgICAgICAgICAgInN0ZXBzIjogc3RlcHNfdmFsLAogICAgICAgICAgICAgICAgImNmZyI6IGNmZ192YWwsCiAgICAgICAgICAgICAgICAic2FtcGxlcl9uYW1lIjogcmVxLnNhbXBsZXJfbmFtZSwKICAgICAgICAgICAgICAgICJzY2hlZHVsZXIiOiByZXEuc2NoZWR1bGVyLAogICAgICAgICAgICAgICAgInBvc2l0aXZlIjogcG9zaXRpdmVfcmVmLAogICAgICAgICAgICAgICAgIm5lZ2F0aXZlIjogbmVnYXRpdmVfcmVmLAogICAgICAgICAgICAgICAgImxhdGVudF9pbWFnZSI6IGxhdGVudF9yZWYsCiAgICAgICAgICAgICAgICAiZGVub2lzZSI6IDEuMCwKICAgICAgICAgICAgfSwKICAgICAgICB9CiAgICAgICAgc2FtcGxlc19yZWYgPSBbIjExIiwgMF0KCiAgICBncmFwaFsiMTIiXSA9IHsKICAgICAgICAiY2xhc3NfdHlwZSI6ICJWQUVEZWNvZGUiLAogICAgICAgICJpbnB1dHMiOiB7CiAgICAgICAgICAgICJzYW1wbGVzIjogc2FtcGxlc19yZWYsCiAgICAgICAgICAgICJ2YWUiOiBbIjQiLCAwXSwKICAgICAgICB9LAogICAgfQogICAgZ3JhcGhbIjEzIl0gPSB7CiAgICAgICAgImNsYXNzX3R5cGUiOiAiU2F2ZUltYWdlIiwKICAgICAgICAiaW5wdXRzIjogewogICAgICAgICAgICAiaW1hZ2VzIjogWyIxMiIsIDBdLAogICAgICAgICAgICAiZmlsZW5hbWVfcHJlZml4IjogcmVxLmZpbGVuYW1lX3ByZWZpeCwKICAgICAgICB9LAogICAgfQogICAgcmV0dXJuIGdyYXBoCgoKZGVmIGJ1aWxkX3JlZmluZV9wcm9tcHRfZ3JhcGgoCiAgICBwcm9tcHQ6IHN0ciwKICAgICosCiAgICBtb2RlOiBzdHIgPSAiZWRpdCIsCiAgICBpbWFnZV9uYW1lczogdHVwbGVbc3RyLCAuLi5dID0gKCksCiAgICBzZWVkOiBpbnQgPSA0MjQyNDIsCiAgICB0aGlua2luZzogYm9vbCA9IFRydWUsCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSA0MDk2LAogICAgc3lzdGVtX3Byb21wdDogc3RyID0gIiIsCikgLT4gZGljdFtzdHIsIGRpY3Rbc3RyLCBBbnldXToKICAgICIiIkJ1aWxkIGEgZGVjb3VwbGVkIFN0YWdlLTEgUHJvbXB0IEVuaGFuY2VyIGdyYXBoIChydW4gd2l0aCBQT1NUIC9mcmVlIGJlZm9yZSBTdGFnZSAyKS4iIiIKICAgIGlmIG1vZGUgbm90IGluICgidDJpIiwgImVkaXQiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zdXBwb3J0ZWQgbW9kZT17bW9kZSFyfS4iKQoKICAgIHBlX2tleSA9ICJwZV9pMmkiIGlmIG1vZGUgPT0gImVkaXQiIGVsc2UgInBlX3QyaSIKICAgIGNsaXBfdHlwZSA9ICJxd2VuX2ltYWdlIiBpZiBtb2RlID09ICJlZGl0IiBlbHNlICJzdGFibGVfZGlmZnVzaW9uIgoKICAgIGdyYXBoOiBkaWN0W3N0ciwgZGljdFtzdHIsIEFueV1dID0gewogICAgICAgICIxIjogewogICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJDTElQTG9hZGVyIiwKICAgICAgICAgICAgImlucHV0cyI6IHsKICAgICAgICAgICAgICAgICJjbGlwX25hbWUiOiBNT0RFTFNbcGVfa2V5XS5maWxlbmFtZSwKICAgICAgICAgICAgICAgICJ0eXBlIjogY2xpcF90eXBlLAogICAgICAgICAgICB9LAogICAgICAgIH0KICAgIH0KCiAgICB0ZXh0Z2VuX2lucHV0czogZGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgImNsaXAiOiBbIjEiLCAwXSwKICAgICAgICAicHJvbXB0IjogcHJvbXB0LAogICAgICAgICJtYXhfbGVuZ3RoIjogaW50KG1heF9sZW5ndGgpLAogICAgICAgICJzYW1wbGluZ19tb2RlIjogIm9uIiwKICAgICAgICAic2FtcGxpbmdfbW9kZS50ZW1wZXJhdHVyZSI6IDEuMCwKICAgICAgICAic2FtcGxpbmdfbW9kZS50b3BfayI6IDIwLAogICAgICAgICJzYW1wbGluZ19tb2RlLnRvcF9wIjogMC45NSwKICAgICAgICAic2FtcGxpbmdfbW9kZS5taW5fcCI6IDAuMCwKICAgICAgICAic2FtcGxpbmdfbW9kZS5yZXBldGl0aW9uX3BlbmFsdHkiOiAxLjAsCiAgICAgICAgInNhbXBsaW5nX21vZGUucHJlc2VuY2VfcGVuYWx0eSI6IDEuNSBpZiBtb2RlID09ICJ0MmkiIGVsc2UgMC4wLAogICAgICAgICJzYW1wbGluZ19tb2RlLnNlZWQiOiBpbnQoc2VlZCksCiAgICAgICAgInRoaW5raW5nIjogYm9vbCh0aGlua2luZyksCiAgICAgICAgInVzZV9kZWZhdWx0X3RlbXBsYXRlIjogbm90IGJvb2woc3lzdGVtX3Byb21wdCksCiAgICAgICAgIm10cCI6ICJhdXRvIiwKICAgIH0KICAgIGlmIHN5c3RlbV9wcm9tcHQ6CiAgICAgICAgdGV4dGdlbl9pbnB1dHNbInN5c3RlbV9wcm9tcHQiXSA9IHN5c3RlbV9wcm9tcHQKCiAgICBuZXh0X2lkID0gMTAKICAgIGZvciBpZHgsIGltZ19uYW1lIGluIGVudW1lcmF0ZShpbWFnZV9uYW1lcywgc3RhcnQ9MSk6CiAgICAgICAgbm9kZV9pZCA9IHN0cihuZXh0X2lkKQogICAgICAgIG5leHRfaWQgKz0gMQogICAgICAgIGdyYXBoW25vZGVfaWRdID0gewogICAgICAgICAgICAiY2xhc3NfdHlwZSI6ICJMb2FkSW1hZ2UiLAogICAgICAgICAgICAiaW5wdXRzIjogeyJpbWFnZSI6IGltZ19uYW1lfSwKICAgICAgICB9CiAgICAgICAgdGV4dGdlbl9pbnB1dHNbZiJpbWFnZXMuaW1hZ2V7aWR4fSJdID0gW25vZGVfaWQsIDBdCgogICAgZ3JhcGhbIjIiXSA9IHsKICAgICAgICAiY2xhc3NfdHlwZSI6ICJUZXh0R2VuZXJhdGVWMiIsCiAgICAgICAgImlucHV0cyI6IHRleHRnZW5faW5wdXRzLAogICAgfQogICAgZ3JhcGhbIjMiXSA9IHsKICAgICAgICAiY2xhc3NfdHlwZSI6ICJQcmV2aWV3QW55IiwKICAgICAgICAiaW5wdXRzIjogeyJzb3VyY2UiOiBbIjIiLCAwXX0sCiAgICB9CiAgICBncmFwaFsiNCJdID0gewogICAgICAgICJjbGFzc190eXBlIjogIlByZXZpZXdBbnkiLAogICAgICAgICJpbnB1dHMiOiB7InNvdXJjZSI6IFsiMiIsIDFdfSwKICAgIH0KICAgIHJldHVybiBncmFwaAo=",
  "bootstrap.py": "...",
  "smoke_test.py": "...",
  "prep_dataset.py": "IiIiQ1BVLW9ubHkgc2NyaXB0IHRvIHN0YWdlIFF3ZW4tSW1hZ2UgMi4xIG1vZGVscyBpbnRvIGEgcHJpdmF0ZSBLYWdnbGUgRGF0YXNldC4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRhcmZpbGUKaW1wb3J0IHVybGxpYi5yZXF1ZXN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55Cgpmcm9tIHF3ZW5fa2FnZ2xlLmJvb3RzdHJhcCBpbXBvcnQgZW5zdXJlX3V2LCB2ZW52X3B5dGhvbl9wYXRoCmZyb20gcXdlbl9rYWdnbGUuY29uc3RhbnRzIGltcG9ydCAoCiAgICBDTE9VREZMQVJFRF9MSU5VWF9VUkwsCiAgICBDT01GWVVJX1JFUE9fVVJMLAogICAgREVGQVVMVF9WRU5WX0RJUiwKICAgIEdQVV9FWFRSQV9QSVBfUEFDS0FHRVMsCiAgICBIRl9SRVBPX0lELAogICAgTU9ERUxTLAogICAgUFJFU0VUUywKICAgIFJVTlRJTUVfQ0FDSEVfU0xVRywKICAgIFJVTlRJTUVfQ0xPVURGTEFSRURfRklMRU5BTUUsCiAgICBSVU5USU1FX0NPTUZZX0FSQ0hJVkUsCiAgICBSVU5USU1FX01FVEFfRklMRU5BTUUsCiAgICBSVU5USU1FX1dIRUVMU19BUkNISVZFLAogICAgVEFFU0RfREVDT0RFUl9GSUxFTkFNRSwKICAgIFRBRVNEX0RFQ09ERVJfVVJMLAogICAgV09SS0ZMT1dfVEVNUExBVEVfVVJMUywKKQoKCmRlZiBfcmVhZF9rYWdnbGVfc2VjcmV0KG5hbWU6IHN0cikgLT4gc3RyOgogICAgdHJ5OgogICAgICAgIGZyb20ga2FnZ2xlX3NlY3JldHMgaW1wb3J0IFVzZXJTZWNyZXRzQ2xpZW50ICAjIHR5cGU6IGlnbm9yZQoKICAgICAgICB2YWwgPSBVc2VyU2VjcmV0c0NsaWVudCgpLmdldF9zZWNyZXQobmFtZSkKICAgICAgICByZXR1cm4gc3RyKHZhbCBvciAiIikuc3RyaXAoKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gIiIKCgpkZWYgX3Jlc29sdmVfa2FnZ2xlX2F1dGgob3duZXI6IHN0ciA9ICIiKSAtPiB0dXBsZVtzdHIsIGRpY3Rbc3RyLCBzdHJdXToKICAgICIiIlJlc29sdmUgS2FnZ2xlIG93bmVyIGhhbmRsZSBhbmQgYXV0aCBlbnYgZm9yIGBrYWdnbGU+PTEuOC4wYCAoYEtBR0dMRV9BUElfVE9LRU5gIG9yIGxlZ2FjeSBrZXkpLiIiIgogICAgYXBpX3Rva2VuID0gb3MuZW52aXJvbi5nZXQoIktBR0dMRV9BUElfVE9LRU4iLCAiIikuc3RyaXAoKSBvciBfcmVhZF9rYWdnbGVfc2VjcmV0KCJLQUdHTEVfQVBJX1RPS0VOIikKICAgIHVzZXJuYW1lID0gb3duZXIuc3RyaXAoKSBvciBvcy5lbnZpcm9uLmdldCgiS0FHR0xFX1VTRVJOQU1FIiwgIiIpLnN0cmlwKCkgb3IgX3JlYWRfa2FnZ2xlX3NlY3JldCgiS0FHR0xFX1VTRVJOQU1FIikKICAgIGxlZ2FjeV9rZXkgPSBvcy5lbnZpcm9uLmdldCgiS0FHR0xFX0tFWSIsICIiKS5zdHJpcCgpIG9yIF9yZWFkX2thZ2dsZV9zZWNyZXQoIktBR0dMRV9LRVkiKQoKICAgIGthZ2dsZV9kaXIgPSBQYXRoLmhvbWUoKSAvICIua2FnZ2xlIgogICAgaWYgbm90IGFwaV90b2tlbjoKICAgICAgICBhY2Nlc3NfdG9rZW5fZmlsZSA9IGthZ2dsZV9kaXIgLyAiYWNjZXNzX3Rva2VuIgogICAgICAgIGlmIGFjY2Vzc190b2tlbl9maWxlLmV4aXN0cygpOgogICAgICAgICAgICBhcGlfdG9rZW4gPSBhY2Nlc3NfdG9rZW5fZmlsZS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3RyaXAoKQoKICAgIGthZ2dsZV9qc29uID0ga2FnZ2xlX2RpciAvICJrYWdnbGUuanNvbiIKICAgIGlmIGthZ2dsZV9qc29uLmV4aXN0cygpIGFuZCAobm90IHVzZXJuYW1lIG9yIChub3QgYXBpX3Rva2VuIGFuZCBub3QgbGVnYWN5X2tleSkpOgogICAgICAgIGRhdGEgPSBqc29uLmxvYWRzKGthZ2dsZV9qc29uLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICB1c2VybmFtZSA9IHVzZXJuYW1lIG9yIHN0cihkYXRhLmdldCgidXNlcm5hbWUiLCAiIikpLnN0cmlwKCkKICAgICAgICBsZWdhY3lfa2V5ID0gbGVnYWN5X2tleSBvciBzdHIoZGF0YS5nZXQoImtleSIsICIiKSkuc3RyaXAoKQoKICAgIGlmIGFwaV90b2tlbjoKICAgICAgICBpZiBub3QgdXNlcm5hbWU6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgICAgICJGb3VuZCBLQUdHTEVfQVBJX1RPS0VOLCBidXQgZGF0YXNldCBtZXRhZGF0YSBhbHNvIG5lZWRzIHlvdXIgS2FnZ2xlIHVzZXJuYW1lLiAiCiAgICAgICAgICAgICAgICAiU2V0IEtBR0dMRV9PV05FUiBpbiB0aGUgbm90ZWJvb2sgY2VsbCBvciBhZGQgS0FHR0xFX1VTRVJOQU1FIGluIEthZ2dsZSBTZWNyZXRzLiIKICAgICAgICAgICAgKQogICAgICAgIHJldHVybiB1c2VybmFtZSwgeyJLQUdHTEVfQVBJX1RPS0VOIjogYXBpX3Rva2VufQoKICAgIGlmIHVzZXJuYW1lIGFuZCBsZWdhY3lfa2V5OgogICAgICAgIHJldHVybiB1c2VybmFtZSwgeyJLQUdHTEVfVVNFUk5BTUUiOiB1c2VybmFtZSwgIktBR0dMRV9LRVkiOiBsZWdhY3lfa2V5fQoKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAiS2FnZ2xlIEFQSSBjcmVkZW50aWFscyBub3QgZm91bmQuIEFkZCBLQUdHTEVfQVBJX1RPS0VOICgrIEtBR0dMRV9PV05FUi9LQUdHTEVfVVNFUk5BTUUpICIKICAgICAgICAib3IgbGVnYWN5IEtBR0dMRV9VU0VSTkFNRSArIEtBR0dMRV9LRVkgaW4gS2FnZ2xlIE5vdGVib29rIFNlY3JldHMuIgogICAgKQoKCmRlZiBfZW5zdXJlX3ByZXBfdmVudih2ZW52X2RpcjogUGF0aCwgKiwgaW5jbHVkZV9waXA6IGJvb2wgPSBGYWxzZSkgLT4gUGF0aDoKICAgIHV2X2JpbiA9IGVuc3VyZV91digpCiAgICBweV9leGUgPSB2ZW52X3B5dGhvbl9wYXRoKHZlbnZfZGlyKQogICAgaWYgbm90IHB5X2V4ZS5leGlzdHMoKToKICAgICAgICBzdWJwcm9jZXNzLnJ1bihbdXZfYmluLCAidmVudiIsICItLXB5dGhvbiIsIHN5cy5leGVjdXRhYmxlLCBzdHIodmVudl9kaXIpXSwgY2hlY2s9VHJ1ZSkKICAgIHBrZ3MgPSBbCiAgICAgICAgImh1Z2dpbmdmYWNlLWh1YltoZl94ZXRdPj0wLjMyLjAiLAogICAgICAgICJrYWdnbGU+PTEuOC4wIiwKICAgICAgICAic2l4IiwKICAgICAgICAid3JhcHQiLAogICAgXQogICAgaWYgaW5jbHVkZV9waXA6CiAgICAgICAgcGtncy5hcHBlbmQoInBpcCIpCiAgICBzdWJwcm9jZXNzLnJ1bigKICAgICAgICBbCiAgICAgICAgICAgIHV2X2JpbiwKICAgICAgICAgICAgInBpcCIsCiAgICAgICAgICAgICJpbnN0YWxsIiwKICAgICAgICAgICAgIi0tcHl0aG9uIiwKICAgICAgICAgICAgc3RyKHB5X2V4ZSksCiAgICAgICAgICAgICpwa2dzLAogICAgICAgIF0sCiAgICAgICAgY2hlY2s9VHJ1ZSwKICAgICkKICAgIHJldHVybiBweV9leGUKCgpkZWYgX3N0YWdlX2ZsYXRfYXNzZXRzKHB5X2V4ZTogUGF0aCwgc3RhZ2luZ19kaXI6IFBhdGgsIHByZXNldDogc3RyKSAtPiBOb25lOgogICAgc3RhZ2luZ19kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZW52ID0gZGljdChvcy5lbnZpcm9uKQogICAgZW52WyJIRl9YRVRfSElHSF9QRVJGT1JNQU5DRSJdID0gIjEiCiAgICBlbnZbIkhGX0hVQl9ESVNBQkxFX1RFTEVNRVRSWSJdID0gIjEiCgogICAgZm9yIGtleSBpbiBQUkVTRVRTW3ByZXNldF06CiAgICAgICAgc3BlYyA9IE1PREVMU1trZXldCiAgICAgICAgZmxhdF90YXJnZXQgPSBzdGFnaW5nX2RpciAvIHNwZWMuZmlsZW5hbWUKICAgICAgICBpZiBmbGF0X3RhcmdldC5leGlzdHMoKSBhbmQgZmxhdF90YXJnZXQuc3RhdCgpLnN0X3NpemUgPiAwOgogICAgICAgICAgICBwcmludChmIltPS10gQWxyZWFkeSBzdGFnZWQ6IHtmbGF0X3RhcmdldC5uYW1lfSIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIHByaW50KGYiW0hGXSBEb3dubG9hZGluZyB7c3BlYy5oZl9wYXRofSAofntzcGVjLmFwcHJveF9naWI6LjJmfSBHaUIpLi4uIiwgZmx1c2g9VHJ1ZSkKICAgICAgICBkbF9jb2RlID0gKAogICAgICAgICAgICAiZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IGhmX2h1Yl9kb3dubG9hZDsgIgogICAgICAgICAgICBmInAgPSBoZl9odWJfZG93bmxvYWQocmVwb19pZD17SEZfUkVQT19JRCFyfSwgZmlsZW5hbWU9e3NwZWMuaGZfcGF0aCFyfSwgbG9jYWxfZGlyPXtzdHIoc3RhZ2luZ19kaXIpIXJ9KTsgIgogICAgICAgICAgICAicHJpbnQocCkiCiAgICAgICAgKQogICAgICAgIHN1YnByb2Nlc3MucnVuKFtzdHIocHlfZXhlKSwgIi1jIiwgZGxfY29kZV0sIGNoZWNrPVRydWUsIGVudj1lbnYpCgogICAgICAgIG5lc3RlZF9maWxlID0gc3RhZ2luZ19kaXIgLyBzcGVjLmhmX3BhdGgKICAgICAgICBpZiBuZXN0ZWRfZmlsZS5leGlzdHMoKToKICAgICAgICAgICAgbmVzdGVkX2ZpbGUucmVwbGFjZShmbGF0X3RhcmdldCkKICAgICAgICBzaHV0aWwucm10cmVlKHN0YWdpbmdfZGlyIC8gIi5jYWNoZSIsIGlnbm9yZV9lcnJvcnM9VHJ1ZSkKCiAgICAjIENsZWFuIG5lc3RlZCBmb2xkZXJzIHNvIHN0YWdpbmdfZGlyIGNvbnRhaW5zIG9ubHkgZmxhdCBhc3NldHMKICAgIGZvciBzdWJkaXJfbmFtZSBpbiAoImRpZmZ1c2lvbl9tb2RlbHMiLCAidGV4dF9lbmNvZGVycyIsICJ2YWUiKToKICAgICAgICBzdWIgPSBzdGFnaW5nX2RpciAvIHN1YmRpcl9uYW1lCiAgICAgICAgaWYgc3ViLmV4aXN0cygpOgogICAgICAgICAgICBzaHV0aWwucm10cmVlKHN1YiwgaWdub3JlX2Vycm9ycz1UcnVlKQoKICAgICMgVEFFU0QgZGVjb2RlcgogICAgdGFlc2RfZHN0ID0gc3RhZ2luZ19kaXIgLyBUQUVTRF9ERUNPREVSX0ZJTEVOQU1FCiAgICBpZiBub3QgKHRhZXNkX2RzdC5leGlzdHMoKSBhbmQgdGFlc2RfZHN0LnN0YXQoKS5zdF9zaXplID4gMCk6CiAgICAgICAgd2l0aCB1cmxsaWIucmVxdWVzdC51cmxvcGVuKFRBRVNEX0RFQ09ERVJfVVJMLCB0aW1lb3V0PTYwKSBhcyByZXNwLCB0YWVzZF9kc3Qub3Blbigid2IiKSBhcyBvdXQ6CiAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihyZXNwLCBvdXQpCgogICAgIyBPZmZpY2lhbCB0ZW1wbGF0ZXMKICAgIGZvciB3Zl9uYW1lLCB3Zl91cmwgaW4gV09SS0ZMT1dfVEVNUExBVEVfVVJMUy5pdGVtcygpOgogICAgICAgIHdmX2RzdCA9IHN0YWdpbmdfZGlyIC8gd2ZfbmFtZQogICAgICAgIGlmIG5vdCAod2ZfZHN0LmV4aXN0cygpIGFuZCB3Zl9kc3Quc3RhdCgpLnN0X3NpemUgPiAwKToKICAgICAgICAgICAgd2l0aCB1cmxsaWIucmVxdWVzdC51cmxvcGVuKHdmX3VybCwgdGltZW91dD02MCkgYXMgcmVzcCwgd2ZfZHN0Lm9wZW4oIndiIikgYXMgb3V0OgogICAgICAgICAgICAgICAgc2h1dGlsLmNvcHlmaWxlb2JqKHJlc3AsIG91dCkKCgpkZWYgX3BhY2tfdGFyX2ZsYXRfcm9vdChzcmNfZGlyOiBQYXRoLCB0YXJfZHN0OiBQYXRoLCAqLCBhcmNfcHJlZml4OiBzdHIgPSAiIikgLT4gTm9uZToKICAgIHRtcF90YXIgPSB0YXJfZHN0LndpdGhfbmFtZSh0YXJfZHN0Lm5hbWUgKyAiLnRtcCIpCiAgICB0cnk6CiAgICAgICAgd2l0aCB0YXJmaWxlLm9wZW4odG1wX3RhciwgbW9kZT0idyIpIGFzIHRmOgogICAgICAgICAgICBmb3IgaXRlbSBpbiBzb3J0ZWQoc3JjX2Rpci5pdGVyZGlyKCkpOgogICAgICAgICAgICAgICAgYXJjbmFtZSA9IGYie2FyY19wcmVmaXh9L3tpdGVtLm5hbWV9IiBpZiBhcmNfcHJlZml4IGVsc2UgaXRlbS5uYW1lCiAgICAgICAgICAgICAgICB0Zi5hZGQoaXRlbSwgYXJjbmFtZT1hcmNuYW1lKQogICAgICAgIHRtcF90YXIucmVwbGFjZSh0YXJfZHN0KQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICB0bXBfdGFyLnVubGluayhtaXNzaW5nX29rPVRydWUpCiAgICAgICAgcmFpc2UKCgpkZWYgc3RhZ2VfcnVudGltZV9jYWNoZSgKICAgIHN0YWdpbmdfZGlyOiBQYXRoLAogICAgKiwKICAgIHdvcmtfZGlyOiBQYXRoID0gUGF0aCgiL3RtcC9xd2VuMjFfcnVudGltZV93b3JrIiksCiAgICB2ZW52X2RpcjogUGF0aCA9IERFRkFVTFRfVkVOVl9ESVIsCiAgICBpbmNsdWRlX3hmb3JtZXJzOiBib29sID0gRmFsc2UsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJTdGFnZSBvZmZsaW5lIEdQVSB3aGVlbHMgKGBxd2VuX3Q0X3doZWVscy50YXJgKSwgYGNvbWZ5dWlfcmVwby50YXJgLCBhbmQgYGNsb3VkZmxhcmVkYCBpbnRvIGZsYXQgYHN0YWdpbmdfZGlyYC4iIiIKICAgIHN0YWdpbmdfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdvcmtfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHB5X2V4ZSA9IF9lbnN1cmVfcHJlcF92ZW52KHZlbnZfZGlyLCBpbmNsdWRlX3BpcD1UcnVlKQoKICAgIGNvbWZ5X3NyYyA9IHdvcmtfZGlyIC8gIkNvbWZ5VUkiCiAgICB3aGVlbHNfZGlyID0gd29ya19kaXIgLyAid2hlZWxzIgogICAgdHJ5OgogICAgICAgIHNodXRpbC5ybXRyZWUoY29tZnlfc3JjLCBpZ25vcmVfZXJyb3JzPVRydWUpCiAgICAgICAgc2h1dGlsLnJtdHJlZSh3aGVlbHNfZGlyLCBpZ25vcmVfZXJyb3JzPVRydWUpCiAgICAgICAgd2hlZWxzX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgICAgIHByaW50KGYiW0NBQ0hFLUJVSUxEXSBDbG9uaW5nIHNoYWxsb3cgQ29tZnlVSSBpbnRvIHtjb21meV9zcmN9Li4uIiwgZmx1c2g9VHJ1ZSkKICAgICAgICBzdWJwcm9jZXNzLnJ1bihbImdpdCIsICJjbG9uZSIsICItLWRlcHRoIiwgIjEiLCBDT01GWVVJX1JFUE9fVVJMLCBzdHIoY29tZnlfc3JjKV0sIGNoZWNrPVRydWUpCgogICAgICAgIGNvbWZ5X3RhciA9IHN0YWdpbmdfZGlyIC8gUlVOVElNRV9DT01GWV9BUkNISVZFCiAgICAgICAgX3BhY2tfdGFyX2ZsYXRfcm9vdChjb21meV9zcmMsIGNvbWZ5X3RhciwgYXJjX3ByZWZpeD0iQ29tZnlVSSIpCiAgICAgICAgcHJpbnQoZiJbQ0FDSEUtQlVJTERdIFBhY2tlZCB7Y29tZnlfdGFyLm5hbWV9ICh7Y29tZnlfdGFyLnN0YXQoKS5zdF9zaXplIC8gKDEwMjQqKjIpOi4xZn0gTWlCKSIsIGZsdXNoPVRydWUpCgogICAgICAgIHJlcV9maWxlID0gY29tZnlfc3JjIC8gInJlcXVpcmVtZW50cy50eHQiCiAgICAgICAgZXh0cmEgPSBsaXN0KEdQVV9FWFRSQV9QSVBfUEFDS0FHRVMpCiAgICAgICAgaWYgaW5jbHVkZV94Zm9ybWVyczoKICAgICAgICAgICAgZXh0cmEuYXBwZW5kKCJ4Zm9ybWVycyIpCiAgICAgICAgcHJpbnQoZiJbQ0FDSEUtQlVJTERdIERvd25sb2FkaW5nIG9mZmxpbmUgd2hlZWxzIGludG8ge3doZWVsc19kaXJ9Li4uIiwgZmx1c2g9VHJ1ZSkKICAgICAgICBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgWwogICAgICAgICAgICAgICAgc3RyKHB5X2V4ZSksCiAgICAgICAgICAgICAgICAiLW0iLAogICAgICAgICAgICAgICAgInBpcCIsCiAgICAgICAgICAgICAgICAiZG93bmxvYWQiLAogICAgICAgICAgICAgICAgIi0tbm8tY2FjaGUtZGlyIiwKICAgICAgICAgICAgICAgICItLWRlc3QiLAogICAgICAgICAgICAgICAgc3RyKHdoZWVsc19kaXIpLAogICAgICAgICAgICAgICAgIi1yIiwKICAgICAgICAgICAgICAgIHN0cihyZXFfZmlsZSksCiAgICAgICAgICAgICAgICAqZXh0cmEsCiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIGNoZWNrPVRydWUsCiAgICAgICAgKQogICAgICAgIHNodXRpbC5ybXRyZWUoY29tZnlfc3JjLCBpZ25vcmVfZXJyb3JzPVRydWUpCiAgICAgICAgd2hlZWxfZmlsZXMgPSBbcCBmb3IgcCBpbiB3aGVlbHNfZGlyLml0ZXJkaXIoKSBpZiBwLmlzX2ZpbGUoKV0KICAgICAgICB3aGVlbHNfdGFyID0gc3RhZ2luZ19kaXIgLyBSVU5USU1FX1dIRUVMU19BUkNISVZFCiAgICAgICAgX3BhY2tfdGFyX2ZsYXRfcm9vdCh3aGVlbHNfZGlyLCB3aGVlbHNfdGFyKQogICAgICAgIHNodXRpbC5ybXRyZWUod2hlZWxzX2RpciwgaWdub3JlX2Vycm9ycz1UcnVlKQogICAgICAgIHByaW50KAogICAgICAgICAgICBmIltDQUNIRS1CVUlMRF0gUGFja2VkIHt3aGVlbHNfdGFyLm5hbWV9OiB7bGVuKHdoZWVsX2ZpbGVzKX0gZmlsZXMgIgogICAgICAgICAgICBmIih7d2hlZWxzX3Rhci5zdGF0KCkuc3Rfc2l6ZSAvICgxMDI0KiozKTouMmZ9IEdpQikiLAogICAgICAgICAgICBmbHVzaD1UcnVlLAogICAgICAgICkKCiAgICAgICAgY2ZfZHN0ID0gc3RhZ2luZ19kaXIgLyBSVU5USU1FX0NMT1VERkxBUkVEX0ZJTEVOQU1FCiAgICAgICAgY2ZfdG1wID0gY2ZfZHN0LndpdGhfbmFtZShjZl9kc3QubmFtZSArICIudG1wIikKICAgICAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4oQ0xPVURGTEFSRURfTElOVVhfVVJMLCB0aW1lb3V0PTkwKSBhcyByZXNwLCBjZl90bXAub3Blbigid2IiKSBhcyBvdXQ6CiAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihyZXNwLCBvdXQsIGxlbmd0aD00ICogMTAyNCAqIDEwMjQpCiAgICAgICAgb3MuY2htb2QoY2ZfdG1wLCAwbzc1NSkKICAgICAgICBjZl90bXAucmVwbGFjZShjZl9kc3QpCgogICAgICAgIG1ldGEgPSB7CiAgICAgICAgICAgICJweXRob25fbW0iOiBmIntzeXMudmVyc2lvbl9pbmZvWzBdfS57c3lzLnZlcnNpb25faW5mb1sxXX0iLAogICAgICAgICAgICAicHl0aG9uX2Z1bGwiOiBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpLAogICAgICAgICAgICAicGxhdGZvcm0iOiBwbGF0Zm9ybS5wbGF0Zm9ybSgpLAogICAgICAgICAgICAibWFjaGluZSI6IHBsYXRmb3JtLm1hY2hpbmUoKSwKICAgICAgICAgICAgIndoZWVsX2NvdW50IjogbGVuKHdoZWVsX2ZpbGVzKSwKICAgICAgICAgICAgImluY2x1ZGVfeGZvcm1lcnMiOiBpbmNsdWRlX3hmb3JtZXJzLAogICAgICAgICAgICAiY3JlYXRlZF9hdF9pc28iOiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKSwKICAgICAgICB9CiAgICAgICAgKHN0YWdpbmdfZGlyIC8gUlVOVElNRV9NRVRBX0ZJTEVOQU1FKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWV0YSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgIHJldHVybiBtZXRhCiAgICBmaW5hbGx5OgogICAgICAgIHNodXRpbC5ybXRyZWUod29ya19kaXIsIGlnbm9yZV9lcnJvcnM9VHJ1ZSkKCgpkZWYgX3VwbG9hZF9wcml2YXRlX2thZ2dsZV9kYXRhc2V0KAogICAgcHlfZXhlOiBQYXRoLAogICAgc3RhZ2luZ19kaXI6IFBhdGgsCiAgICAqLAogICAgdXNlcm5hbWU6IHN0ciwKICAgIGF1dGhfZW52OiBkaWN0W3N0ciwgc3RyXSwKICAgIGRhdGFzZXRfc2x1Zzogc3RyLAogICAgdGl0bGU6IHN0ciwKICAgIHVwZGF0ZV9leGlzdGluZzogYm9vbCwKICAgIHZlcnNpb25fbm90ZXM6IHN0ciwKKSAtPiBzdHI6CiAgICBkYXRhc2V0X2lkID0gZiJ7dXNlcm5hbWV9L3tkYXRhc2V0X3NsdWd9IgogICAgbWV0YSA9IHsKICAgICAgICAidGl0bGUiOiB0aXRsZSwKICAgICAgICAiaWQiOiBkYXRhc2V0X2lkLAogICAgICAgICJsaWNlbnNlcyI6IFt7Im5hbWUiOiAib3RoZXIifV0sCiAgICB9CiAgICAoc3RhZ2luZ19kaXIgLyAiZGF0YXNldC1tZXRhZGF0YS5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKG1ldGEsIGluZGVudD0yKSwKICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgKQoKICAgIGthZ2dsZV9jbGkgPSBweV9leGUucGFyZW50IC8gKCJrYWdnbGUuZXhlIiBpZiBvcy5uYW1lID09ICJudCIgZWxzZSAia2FnZ2xlIikKICAgIGVudiA9IGRpY3Qob3MuZW52aXJvbikKICAgIGZvciBrIGluICgiS0FHR0xFX0FQSV9UT0tFTiIsICJLQUdHTEVfVVNFUk5BTUUiLCAiS0FHR0xFX0tFWSIpOgogICAgICAgIGVudi5wb3AoaywgTm9uZSkKICAgIGVudi51cGRhdGUoYXV0aF9lbnYpCgogICAgaWYgdXBkYXRlX2V4aXN0aW5nOgogICAgICAgIGNtZCA9IFsKICAgICAgICAgICAgc3RyKGthZ2dsZV9jbGkpLAogICAgICAgICAgICAiZGF0YXNldHMiLAogICAgICAgICAgICAidmVyc2lvbiIsCiAgICAgICAgICAgICItcCIsCiAgICAgICAgICAgIHN0cihzdGFnaW5nX2RpciksCiAgICAgICAgICAgICItbSIsCiAgICAgICAgICAgIHZlcnNpb25fbm90ZXMsCiAgICAgICAgICAgICItLWRpci1tb2RlIiwKICAgICAgICAgICAgInNraXAiLAogICAgICAgIF0KICAgIGVsc2U6CiAgICAgICAgY21kID0gWwogICAgICAgICAgICBzdHIoa2FnZ2xlX2NsaSksCiAgICAgICAgICAgICJkYXRhc2V0cyIsCiAgICAgICAgICAgICJjcmVhdGUiLAogICAgICAgICAgICAiLXAiLAogICAgICAgICAgICBzdHIoc3RhZ2luZ19kaXIpLAogICAgICAgICAgICAiLS1kaXItbW9kZSIsCiAgICAgICAgICAgICJza2lwIiwKICAgICAgICBdCgogICAgcHJpbnQoZiJbS0FHR0xFXSBVcGxvYWRpbmcgUFJJVkFURSBkYXRhc2V0IHtkYXRhc2V0X2lkfSBmcm9tIHtzdGFnaW5nX2Rpcn0uLi4iLCBmbHVzaD1UcnVlKQogICAgc3VicHJvY2Vzcy5ydW4oY21kLCBjaGVjaz1UcnVlLCBlbnY9ZW52KQogICAgcHJpbnQoZiJbRE9ORV0gRGF0YXNldCByZWFkeTogaHR0cHM6Ly93d3cua2FnZ2xlLmNvbS9kYXRhc2V0cy97ZGF0YXNldF9pZH0iLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIGRhdGFzZXRfaWQKCgpkZWYgYnVpbGRfYW5kX3VwbG9hZF9kYXRhc2V0KAogICAgKiwKICAgIGRhdGFzZXRfc2x1Zzogc3RyID0gInF3ZW4taW1hZ2UtMjEtaW50OC1jb252cm90IiwKICAgIHRpdGxlOiBzdHIgPSAiUXdlbi1JbWFnZSAyLjEgSU5UOCBDb252Um90IFdlaWdodHMiLAogICAgcHJlc2V0OiBzdHIgPSAiZnVsbF9pMmkiLAogICAgb3duZXI6IHN0ciA9ICIiLAogICAgc3RhZ2luZ19kaXI6IFBhdGggPSBQYXRoKCIvdG1wL3F3ZW4yMV9kYXRhc2V0X3N0YWdpbmciKSwKICAgIHZlbnZfZGlyOiBQYXRoID0gREVGQVVMVF9WRU5WX0RJUiwKICAgIHVwZGF0ZV9leGlzdGluZzogYm9vbCA9IEZhbHNlLAopIC0+IHN0cjoKICAgICIiIkRvd25sb2FkIHdlaWdodHMgdG8gL3RtcCBvbiBDUFUgYW5kIHB1c2ggYSBQUklWQVRFIEthZ2dsZSBEYXRhc2V0IChmbGF0IGxheW91dCkuIiIiCiAgICBpZiBwcmVzZXQgbm90IGluIFBSRVNFVFM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gcHJlc2V0PXtwcmVzZXQhcn07IGV4cGVjdGVkIG9uZSBvZiB7dHVwbGUoUFJFU0VUUyl9IikKCiAgICB1c2VybmFtZSwgYXV0aF9lbnYgPSBfcmVzb2x2ZV9rYWdnbGVfYXV0aChvd25lcj1vd25lcikKICAgIHB5X2V4ZSA9IF9lbnN1cmVfcHJlcF92ZW52KHZlbnZfZGlyKQogICAgX3N0YWdlX2ZsYXRfYXNzZXRzKHB5X2V4ZSwgc3RhZ2luZ19kaXIsIHByZXNldCkKICAgIHJldHVybiBfdXBsb2FkX3ByaXZhdGVfa2FnZ2xlX2RhdGFzZXQoCiAgICAgICAgcHlfZXhlLAogICAgICAgIHN0YWdpbmdfZGlyLAogICAgICAgIHVzZXJuYW1lPXVzZXJuYW1lLAogICAgICAgIGF1dGhfZW52PWF1dGhfZW52LAogICAgICAgIGRhdGFzZXRfc2x1Zz1kYXRhc2V0X3NsdWcsCiAgICAgICAgdGl0bGU9dGl0bGUsCiAgICAgICAgdXBkYXRlX2V4aXN0aW5nPXVwZGF0ZV9leGlzdGluZywKICAgICAgICB2ZXJzaW9uX25vdGVzPWYiVXBkYXRlIFF3ZW4tSW1hZ2UgMi4xIHByZXNldD17cHJlc2V0fSIsCiAgICApCgoKZGVmIGJ1aWxkX2FuZF91cGxvYWRfcnVudGltZV9jYWNoZSgKICAgICosCiAgICBkYXRhc2V0X3NsdWc6IHN0ciA9IFJVTlRJTUVfQ0FDSEVfU0xVRywKICAgIHRpdGxlOiBzdHIgPSAiUXdlbi1JbWFnZSAyLjEgVDQgUnVudGltZSBDYWNoZSAoV2hlZWxzICsgQ29tZnlVSSArIGNsb3VkZmxhcmVkKSIsCiAgICBvd25lcjogc3RyID0gIiIsCiAgICBzdGFnaW5nX2RpcjogUGF0aCA9IFBhdGgoIi90bXAvcXdlbjIxX3J1bnRpbWVfc3RhZ2luZyIpLAogICAgd29ya19kaXI6IFBhdGggPSBQYXRoKCIvdG1wL3F3ZW4yMV9ydW50aW1lX3dvcmsiKSwKICAgIHZlbnZfZGlyOiBQYXRoID0gREVGQVVMVF9WRU5WX0RJUiwKICAgIHVwZGF0ZV9leGlzdGluZzogYm9vbCA9IEZhbHNlLAogICAgaW5jbHVkZV94Zm9ybWVyczogYm9vbCA9IEZhbHNlLAopIC0+IHN0cjoKICAgICIiIlN0YWdlIG9mZmxpbmUgUHl0aG9uIHdoZWVscyArIENvbWZ5VUkgcmVwbyArIGNsb3VkZmxhcmVkIG9uIENQVSBhbmQgdXBsb2FkIGFzIGEgUFJJVkFURSBLYWdnbGUgRGF0YXNldC4iIiIKICAgIHVzZXJuYW1lLCBhdXRoX2VudiA9IF9yZXNvbHZlX2thZ2dsZV9hdXRoKG93bmVyPW93bmVyKQogICAgcHlfZXhlID0gX2Vuc3VyZV9wcmVwX3ZlbnYodmVudl9kaXIsIGluY2x1ZGVfcGlwPVRydWUpCiAgICBzdGFnZV9ydW50aW1lX2NhY2hlKAogICAgICAgIHN0YWdpbmdfZGlyLAogICAgICAgIHdvcmtfZGlyPXdvcmtfZGlyLAogICAgICAgIHZlbnZfZGlyPXZlbnZfZGlyLAogICAgICAgIGluY2x1ZGVfeGZvcm1lcnM9aW5jbHVkZV94Zm9ybWVycywKICAgICkKICAgIHJldHVybiBfdXBsb2FkX3ByaXZhdGVfa2FnZ2xlX2RhdGFzZXQoCiAgICAgICAgcHlfZXhlLAogICAgICAgIHN0YWdpbmdfZGlyLAogICAgICAgIHVzZXJuYW1lPXVzZXJuYW1lLAogICAgICAgIGF1dGhfZW52PWF1dGhfZW52LAogICAgICAgIGRhdGFzZXRfc2x1Zz1kYXRhc2V0X3NsdWcsCiAgICAgICAgdGl0bGU9dGl0bGUsCiAgICAgICAgdXBkYXRlX2V4aXN0aW5nPXVwZGF0ZV9leGlzdGluZywKICAgICAgICB2ZXJzaW9uX25vdGVzPWYiVXBkYXRlIFQ0IHJ1bnRpbWUgY2FjaGUgKHB5IHtzeXMudmVyc2lvbl9pbmZvWzBdfS57c3lzLnZlcnNpb25faW5mb1sxXX0pIiwKICAgICkKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iU3RhZ2UgUXdlbi1JbWFnZSAyLjEgd2VpZ2h0cyBvciBydW50aW1lIGNhY2hlIGludG8gcHJpdmF0ZSBLYWdnbGUgRGF0YXNldHMuIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdGFyZ2V0IiwgZGVmYXVsdD0id2VpZ2h0cyIsIGNob2ljZXM9KCJ3ZWlnaHRzIiwgInJ1bnRpbWVfY2FjaGUiLCAiYm90aCIpKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zbHVnIiwgZGVmYXVsdD0icXdlbi1pbWFnZS0yMS1pbnQ4LWNvbnZyb3QiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10aXRsZSIsIGRlZmF1bHQ9IlF3ZW4tSW1hZ2UgMi4xIElOVDggQ29udlJvdCBXZWlnaHRzIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcnVudGltZS1zbHVnIiwgZGVmYXVsdD1SVU5USU1FX0NBQ0hFX1NMVUcpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByZXNldCIsIGRlZmF1bHQ9ImZ1bGxfaTJpIiwgY2hvaWNlcz10dXBsZShQUkVTRVRTKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3duZXIiLCBkZWZhdWx0PSIiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zdGFnaW5nLWRpciIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCIvdG1wL3F3ZW4yMV9kYXRhc2V0X3N0YWdpbmciKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcnVudGltZS1zdGFnaW5nLWRpciIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCIvdG1wL3F3ZW4yMV9ydW50aW1lX3N0YWdpbmciKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdmVudi1kaXIiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9REVGQVVMVF9WRU5WX0RJUikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdXBkYXRlIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgaWYgYXJncy50YXJnZXQgaW4gKCJ3ZWlnaHRzIiwgImJvdGgiKToKICAgICAgICBidWlsZF9hbmRfdXBsb2FkX2RhdGFzZXQoCiAgICAgICAgICAgIGRhdGFzZXRfc2x1Zz1hcmdzLnNsdWcsCiAgICAgICAgICAgIHRpdGxlPWFyZ3MudGl0bGUsCiAgICAgICAgICAgIHByZXNldD1hcmdzLnByZXNldCwKICAgICAgICAgICAgb3duZXI9YXJncy5vd25lciwKICAgICAgICAgICAgc3RhZ2luZ19kaXI9YXJncy5zdGFnaW5nX2RpciwKICAgICAgICAgICAgdmVudl9kaXI9YXJncy52ZW52X2RpciwKICAgICAgICAgICAgdXBkYXRlX2V4aXN0aW5nPWFyZ3MudXBkYXRlLAogICAgICAgICkKICAgIGlmIGFyZ3MudGFyZ2V0IGluICgicnVudGltZV9jYWNoZSIsICJib3RoIik6CiAgICAgICAgYnVpbGRfYW5kX3VwbG9hZF9ydW50aW1lX2NhY2hlKAogICAgICAgICAgICBkYXRhc2V0X3NsdWc9YXJncy5ydW50aW1lX3NsdWcsCiAgICAgICAgICAgIG93bmVyPWFyZ3Mub3duZXIsCiAgICAgICAgICAgIHN0YWdpbmdfZGlyPWFyZ3MucnVudGltZV9zdGFnaW5nX2RpciwKICAgICAgICAgICAgdmVudl9kaXI9YXJncy52ZW52X2RpciwKICAgICAgICAgICAgdXBkYXRlX2V4aXN0aW5nPWFyZ3MudXBkYXRlLAogICAgICAgICkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
  "tpu_runtime.py": "...",
  "gateway.py": "..."
}

for name, b64 in _MODULES_B64.items():
    (PKG_DIR / name).write_bytes(base64.b64decode(b64))

for mod_key in [k for k in list(sys.modules) if k == 'qwen_kaggle' or k.startswith('qwen_kaggle.')]:
    sys.modules.pop(mod_key, None)

if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

print(f'Unpacked {len(_MODULES_B64)} modules into {PKG_DIR}')


In [ ]:
import argparse, json
from pathlib import Path
from qwen_kaggle.constants import DEFAULT_COMFY_DIR, DEFAULT_INPUT_DIR, DEFAULT_VENV_DIR, DEFAULT_WORKING_DIR
from qwen_kaggle.smoke_test import run_smoke_suite

# --- Opcje testu ---
TEST_BASE = False       # True: pobierze i przetestuje tez Base INT8 (25 krokow)
TEST_XFORMERS = False   # True: doinstaluje xformers do porownania z PyTorch SDPA
TEST_FP16_VAE = False   # True: wymusi --fp16-vae (domyslnie testujemy bezpieczne VAE)
TEST_DUAL_GPU = True    # True: sprawdzi rowniez osobny proces na GPU 1
FORCE_HF = False        # True: ignoruj /kaggle/input i pobierz z HuggingFace

args = argparse.Namespace(
    comfy_dir=DEFAULT_COMFY_DIR,
    venv_dir=DEFAULT_VENV_DIR,
    input_dir=DEFAULT_INPUT_DIR,
    out_dir=DEFAULT_WORKING_DIR / 'smoke_test_results',
    force_hf=FORCE_HF,
    xformers=TEST_XFORMERS,
    fp16_vae=TEST_FP16_VAE,
    pytorch_sdpa=False,
    test_base=TEST_BASE,
    test_dual_gpu=TEST_DUAL_GPU,
)
report = run_smoke_suite(args)
print(json.dumps({
    'peak_ram_used_gb': report['peak_ram_used_gb'],
    'peak_gpu_vram_mb': report['peak_gpu_vram_mb'],
    'results': report['results'],
}, indent=2))


In [ ]:
from pathlib import Path
from IPython.display import Image, display

res_dir = Path('/kaggle/working/smoke_test_results')
for png in sorted(res_dir.glob('*.png')):
    if png.name == 'smoke_edit_input.png':
        continue
    print(png.name)
    display(Image(filename=str(png), width=384))
